# transformer-residual-lab · Kaggle 低位宽量化验证台

**版本** v0.3.1 · **目标 checkpoint** `Qwen/Qwen3-8B`（8.19 B 参数，36 层，hidden 4096，GQA 32/8，head_dim 128，vocab 151936）

在**单张 T4 16 GB** 上，用**同一份 base GGUF + 同一份 imatrix**产出 `IQ2_XS` 与 `IQ4_XS`，然后比较**真实文件字节**、**配对 NLL**、**有限的配对质量指标**。

## 三步上手（网页操作，不碰本地终端）

1. 右上角 `Edit` → 找到 `RUN_S2 = False` 这一行，改成 `RUN_S2 = True`（只想先看环境就跳过这步）。
2. 右侧 `Settings` → `Internet` 设为 **On**，`Accelerator` 选 **GPU T4 ×2**。
3. 菜单 `Run` → **Run all**。约 15–30 分钟（含编译）；日志和产物都列在最后。

`RUN_S2 = False` 时只跑 `S0_ENV` / `S1_SMOKE` / `S9_REPORT`，约 2–4 分钟，**不联网、不下载任何东西**。

**唯一会联网的地方是 S2**：下载固定 commit 的 llama.cpp 源码 tarball（36.6 MiB，sha256 预登记并强制校验）。它**不会**下载模型权重、不会 `pip install`、不会租 GPU、不会上传任何本地文件。

---

## v0.3 改了什么（一键 S2）

| ID | 问题 | 修法 |
|---|---|---|
| **B-1** | 让用户在 Mac 上下载源码、解压、传 36.6 MiB Dataset、填绝对路径 —— 纯人工负担 | 新增 `RUN_S2` 一个开关：自动下载 pinned commit 的 tarball → 校验 sha256 → 解压到 `WORK/src/` → 编译。用户不碰本地 |
| **B-2** | 旧 `ALLOW_NETWORK=True` 走 `git clone --depth 1`（HEAD），与固定 `LLAMA_COMMIT` 直接冲突 —— "固定版本"这句话失效 | **删掉该分支**。联网只有一条路径：pinned tarball + 预登记 sha256 |
| **B-3** | 默认行为不明确：既想"一键"又不想"导入就下载" | `RUN_S2 = False` 是默认值；`_apply_run_switches()` 在 False 时**一个字段都不改**，并有测试钉住 |
| **B-4** | S2 失败时用户面对几十行 cmake 输出 | 失败时给"症状 → 处置"的一句话结论 + 尾部日志路径；S9 报告里单列一节记录 S2 状态 |
| **C-1** | 真实 Kaggle 失败：`CUDA::cuda_driver` IMPORTED target 找不到，`ggml-cuda/CMakeLists.txt:181` CMake Error | 默认传 `-DGGML_CUDA_NO_VMM=ON`（跳过 libcuda 直链与 VMM 显存池，算力内核不变），仍保留 `GGML_CUDA=ON` + sm_75；`CUDA_NO_VMM` 进 S2 指纹 |
| **C-2** | 上面的错误被误判成"找不到 nvcc"（全文包含判断：日志里"nvcc"与"not found"相隔十几行） | 分类器改**同行匹配** + driver 专用分支，排在 nvcc 分支之前 |

---

## v0.2 修了什么（对应 Issue #5 的七条）

| ID | 问题 | 修法 |
|---|---|---|
| **P0-A** | `build = src / "build"` 会往只读的 `/kaggle/input` 写 | build 目录固定为 `WORK/build-llama`，`cmake -S <src> -B <build>`；`llama_bin()` 增加该路径；`CFG["LLAMA_COMMIT"]` 可预注册源码 commit |
| **P0-B** | `size + 1 << 30` 被解析成 `(size+1)<<30`，需求被夸大约 **9.5 亿倍** | 改为 `_est_need_bytes()` 纯函数，显式括号；8 GiB / 16 GiB 各一条单元测试 |
| **P0-C** | S9 在 bf16 失败时 `tb["bf16"]["tflops"]` 直接 `KeyError` | 抽出 `_bench_summary()`，全部 `.get()`；保留 bf16 的 error 文本；mock 测试覆盖 |
| **P1-D** | 断点只看 `status=="ok"`，换 base/imatrix 也跳过 | 每阶段**输入指纹**（CFG 子集 + 文件 size/mtime/首尾 1 MiB 哈希 + 源码 commit + notebook 版本）；指纹变了就失效重跑；**S9 每次重新生成** |
| **P1-E** | `run_cmd(check=False)` 让失败命令的残留旧文件被当成功 | `run_cmd` 一律在 rc≠0 时抛 `CommandFailed`；产物先写 `.partial`，验完再 `os.replace` 原子改名 |
| **P1-F** | `split_eval` 的 `ceil` 不保证组数；组间 token 数不等；VRAM 只查入口 | 精确组数（整除+余数分配）；记录每组字符数并给字符加权均值；`guard()` 明写"只做入口检查，不能阻止运行中 OOM" |
| **P1-G** | 未知 ggml type 被当成 8 bpw 正常出表 | 未知 type 抛 `UnknownGGMLType`，S5 记 `byte_breakdown_available: False`，不产出错误 bpw |

---

## 四条不可越界的红线

| 事实 | 后果 |
|---|---|
| `llama-quant.cpp:882` 把 `GGML_TYPE_IQ2_XS` 列进 `tensor_requires_imatrix`；`:1324` 缺 imatrix 时 `throw std::runtime_error("The result will be garbage, so bailing out")` | `llama-quantize ... IQ2_XS` **直接退出、不产出文件**。S4 把 imatrix 设为硬前置，缺失即报错，不降级 |
| `IQ4_XS` **不**要求 imatrix，`IQ2_XS` 要求 | 只给 Q2 配就是给低比特开小灶 → **两者共用同一份 `imatrix.gguf`** |
| imatrix 会改变 `llama_tensor_get_type` 的分层决策（`llama-quant.cpp:688` 按 `has_imatrix` 分支） | 两个文件实际张量类型组合不同 → **字节必须从文件读**，不能用名义 bpw 乘参数 |
| T4 是 Turing（CC 7.5），**无原生 bf16**；8B f16 权重 15.25 GiB > 可用约 15.0 GiB | 高精度锚点在 T4 上跑不了。S8 先算显存再决定，默认返回 `skipped` 并给两个数字；Q8_0 只算**近似参照**，不构成"相对 BF16"的有效性结论 |


## 阶段总览

| 阶段 | 默认 | 前置依赖 | 预算 | 产出 |
|---|---|---|---|---|
| `S0_ENV` | **开** | — | 120 s | GPU/CPU/RAM/磁盘/工具链 + `/kaggle/input` 里所有 `.gguf` 的完整路径 |
| `S1_SMOKE` | **开** | — | 240 s | 合成 GGUF 往返自检、fp16/bf16 矩阵乘、拷贝带宽、磁盘写速 |
| `S2_BUILD` | 关（`RUN_S2=True` 自动开） | **无输入**：源码由 S2 自己下载并校验 | 2400 s | `llama-imatrix` / `llama-quantize` / `llama-perplexity` / `llama-cli`，build 在 `WORK/build-llama` |
| `S3_IMATRIX` | 关 | S2 + base GGUF + 校准语料 | 3600 s | `imatrix.gguf`（原子写入）+ 指纹 |
| `S4_QUANT` | 关 | S2 + S3 + base GGUF | 1800 s | 两个 GGUF（`.partial` → 原子改名）+ 逐文件 bpw |
| `S5_BYTES` | 关 | S4（或任意 GGUF） | 300 s | 逐张量类型字节、按 attn/ffn/embed 分类；未知 type 则拒绝出表 |
| `S6_NLL` | 关 | S2 + S5 + 评测语料 | 5400 s | 分组配对 NLL + bootstrap CI（pilot 级） |
| `S7_QUALITY` | 关 | S2 + S5 | 1800 s | 贪心续写一致性、repeat4 |
| `S8_BF16_ANCHOR` | 关 | 高精度 GGUF + 足够显存 | 3600 s | 锚点 NLL，或一条写明 `required_gb` / `vram_usable_gb` 的 skip 记录 |
| `SX_EXPLORE` | 关 | 第三方 GGUF | 1800 s | 字节 + NLL，标 `provenance.fair_comparison=False` |
| `S9_REPORT` | **开** | — | 120 s | `report.md` + `results.json`（**每次重新生成，从不 RESUME**） |

---

## 你需要先准备什么

手动挂载为 Kaggle Dataset（只读挂在 `/kaggle/input/<slug>/`）：

| # | 内容 | 建议 | 大小 |
|---|---|---|---|
| 1 | **base GGUF** | `Qwen3-8B` 的 f16 或 Q8_0 | f16 = 15.25 GiB；Q8_0 ≈ 8.1 GiB |
| 2 | **校准语料** `calib.txt` | wikitext-2-raw **train** | ≥ 200 KB |
| 3 | **评测语料** `eval.txt` | wikitext-2-raw **test**（必须不同 split） | 几十 KB 起 |
| 4 | **llama.cpp 源码** | 近期 master | ~50 MB |
| 5 | （可选）第三方量化 GGUF | 别人的 IQ2_XS / IQ4_XS | 2–5 GiB |

**两个 base 不是同一个东西（v0.2.1）**：

| CFG 键 | 谁用 | 要求 |
|---|---|---|
| `BASE_GGUF` / `BASE_KIND` | **S4 量化**、S8 锚点 | **必须是 f16/bf16**。已量化的 base 再量化需要 `--allow-requantize`，官方帮助原文是 `WARNING: this can severely reduce quality compared to quantizing from 16bit or 32bit!` |
| `IMATRIX_BASE` / `IMATRIX_BASE_KIND` | **S3 算 imatrix** | 留空 = 同 `BASE_GGUF`。T4 上可以指向 Q8_0（能进显存，快几十倍） |

**T4 上的推荐配置**：`IMATRIX_BASE` = Q8_0（imatrix 需要整模型进显存），`BASE_GGUF` = 同 checkpoint 的 f16（`llama-quantize` 是 CPU 路径，15.25 GiB 走的是内存不是显存）。两者必须来自同一 checkpoint —— S4 会核对 architecture 与每个张量的形状，不一致直接报错。

如果只有 Q8_0 一个文件、又只想做 smoke，必须显式设 `CFG["ALLOW_REQUANTIZE"] = True`，结果会标 `requantized_pilot_only`，**不能**当作相对原始 checkpoint 的高保真低比特结论。

---

## 关于断点恢复的新语义（v0.2）

旧版只看 `status == "ok"`。现在每个阶段存一个**输入指纹**：

```
fingerprint = sha256( notebook_version
                    + stage 依赖的 CFG 子集
                    + 每个输入文件的 (size, mtime_ns, sha256(head 1MiB), sha256(tail 1MiB))
                    + 源码 git commit )
```

- 指纹相同 → `RESUME`
- 指纹不同 → `INVALIDATE`，重跑该阶段（**不需要** `FORCE_RERUN=True` 把前置全部重算）
- `S9_REPORT` 在 `ALWAYS_RERUN` 里，**永远重新生成**，保证汇总不会停留在旧结果

大文件不做全量 sha256（2.4 GB 要 10 s+），用 size + mtime + 首尾各 1 MiB 的组合。这不足以对抗刻意伪造，但足以捕捉"换了 base / 换了 imatrix / 改了 chunk 数"这类真实误用。


In [ ]:
# =====================================================================
# 你只需要看这一行
# =====================================================================
# RUN_S2 = False  →  只跑 S0 / S1 / S9（约 2–4 分钟，不联网、不下载任何东西）。
# RUN_S2 = True   →  额外下载固定 commit 的 llama.cpp 源码（**只有这一个文件，
#                    36.6 MiB**），校验 sha256，解压到 /kaggle/working，然后编译
#                    llama-imatrix / llama-quantize / llama-perplexity / llama-cli。
#
# 打开 RUN_S2 之前请确认 Notebook 右侧 Settings → Internet = On。
# 它**不会**下载模型权重、不会 pip install、不会租 GPU、不会上传任何本地文件。
RUN_S2 = False

# =====================================================================
# 下面是派生配置。RUN_S2=True 时全部自动填好，不需要你改。
# =====================================================================
from pathlib import Path

NOTEBOOK_VERSION = "0.3.1"

# 固定 commit 的源码。sha256 是 **tarball 的**，不是解压后目录的。
# 换版本必须同时换 commit / url / sha256 / bytes / dir 五项，否则 S2 会拒绝解压。
S2_PINNED = {
    "commit": "d454a204ef2b63af357744809c05af897611a72e",
    "url": ("https://github.com/ggml-org/llama.cpp/archive/"
            "d454a204ef2b63af357744809c05af897611a72e.tar.gz"),
    "sha256": "f565ee976503e0b66997a05554d4881a252ced7687811f2a41df771203b54d7f",
    "bytes": 38419835,
    "dir": "llama.cpp-d454a204ef2b63af357744809c05af897611a72e",
}

STAGES = {
    "S0_ENV":         True,    # 环境 / GPU / 磁盘检查（秒级）
    "S1_SMOKE":       True,    # 短时 smoke：合成 GGUF 往返 + 显存/磁盘微基准
    "S2_BUILD":       False,   # 由 RUN_S2 控制；手改这里也行，但 RUN_S2 更省事
    "S3_IMATRIX":     False,   # 生成 imatrix.gguf
    "S4_QUANT":       False,   # 产出 IQ2_XS / IQ4_XS（共用同一份 imatrix）
    "S5_BYTES":       False,   # 读文件算真实字节与 bpw
    "S6_NLL":         False,   # 配对 NLL + bootstrap CI
    "S7_QUALITY":     False,   # 有限配对质量检查
    "S8_BF16_ANCHOR": False,   # 高精度锚点（T4 上会主动拒绝）
    "S9_REPORT":      True,    # 汇总（每次重新生成）
    "SX_EXPLORE":     False,   # 第三方现成量化，仅探索
}

# 这些阶段即使 status=="ok" 也重跑
ALWAYS_RERUN = {"S9_REPORT"}

CFG = {
    # ---------- 模型 ----------
    "MODEL_ID":    "Qwen/Qwen3-8B",
    "PARAM_COUNT": 8.19e9,
    "N_LAYERS":    36,
    "HIDDEN":      4096,
    "N_KV_HEADS":  8,
    "HEAD_DIM":    128,

    # ---------- 文件路径（S3 起需要，全部手动挂载到 /kaggle/input） ----------
    # BASE_GGUF = 量化 base。**正式路径必须是 f16/bf16**：
    #   从已量化的 base 再量化，llama-quantize 需要 --allow-requantize，
    #   且官方帮助明确警告 "can severely reduce quality"。
    "BASE_GGUF":    "",   # 例 "/kaggle/input/qwen3-8b-gguf/qwen3-8b-f16.gguf"
    "BASE_KIND":    "",   # 必须与文件一致："f16" / "bf16" / "q8_0" / "q4_k_m" ...
    # IMATRIX_BASE = 算 imatrix 用的 base，留空则与 BASE_GGUF 相同。
    #   在 T4 上可以让它指向 Q8_0（能进显存，快），而 BASE_GGUF 仍是 f16。
    #   两者必须来自同一 checkpoint（S4 会核对 architecture + 张量形状）。
    "IMATRIX_BASE": "",
    "IMATRIX_BASE_KIND": "",  # 留空则用 BASE_KIND
    "IMATRIX_GGUF": "",   # 留空则 S3 生成到 working。**不要指向 /kaggle/input（只读）**
    "CALIB_TXT":    "",   # imatrix 校准语料
    "EVAL_TXT":     "",   # NLL 评测语料（必须与校准集不同源）
    "EVAL_SOURCE":  "",   # 例 "wikitext-2-raw/test"
    "CALIB_SOURCE": "",   # 例 "wikitext-2-raw/train"
    "EXPLORE_GGUF": "",   # 第三方现成量化，仅探索

    # ---------- llama.cpp ----------
    "LLAMA_SRC":    "",      # 手动源码目录；留空则自动获取（RUN_S2=True）或找 /kaggle/input
    "LLAMA_BIN":    "",      # 已有构建的 bin 目录；留空则用 WORK/build-llama/bin
    "LLAMA_COMMIT": "",      # RUN_S2=True 时自动填成 S2_PINNED["commit"]
    "S2_AUTO_FETCH_SOURCE": False,  # 由 RUN_S2 打开。True 才允许联网取源码
    "BUILD_CUDA":   True,    # False 则只构建 CPU 版（imatrix / quantize 都是 CPU 路径）
    "CUDA_ARCH":    "75",    # T4 = sm_75。留空则用上游默认的多架构列表（4 核上慢数倍）
    # v0.3.1：Kaggle T4 上 find_package(CUDAToolkit) 解析不出 CUDA::cuda_driver
    # IMPORTED target，ggml-cuda 的 target_link_libraries 直接 CMake Error。
    # GGML_CUDA_NO_VMM=ON 只跳过 libcuda 直链 + VMM 显存池（回退 cudaMalloc 池），
    # 算力内核不变。见 ggml/src/ggml-cuda/CMakeLists.txt:178-182 与 common.cuh:265。
    "CUDA_NO_VMM":  True,
    "BUILD_JOBS":   4,

    # ---------- 预算（硬上限，超了主动抛异常而不是被 OOM kill） ----------
    "MAX_WALL_SEC":      8 * 3600,
    "MAX_VRAM_GB":       14.0,
    "MIN_FREE_DISK_GB":  3.0,
    "STAGE_BUDGET_SEC": {
        "S0_ENV": 120, "S1_SMOKE": 240, "S2_BUILD": 2400,
        "S3_IMATRIX": 3600, "S4_QUANT": 1800, "S5_BYTES": 300,
        "S6_NLL": 5400, "S7_QUALITY": 1800, "S8_BF16_ANCHOR": 3600,
        "S9_REPORT": 120, "SX_EXPLORE": 1800,
    },

    # ---------- 实验规模（首版刻意保守） ----------
    "IMATRIX_CTX":           512,
    "IMATRIX_CHUNKS":        32,
    "NLL_CTX":               2048,
    "NLL_CHUNKS_PER_GROUP":  4,
    "NLL_GROUPS":            8,      # pilot 级；正式 CI 需按独立文档重新设计
    "QUALITY_PROMPTS":       8,
    "QUALITY_N_PREDICT":     32,

    # ---------- 锚点 ----------
    "ANCHOR_EXTRA_GB":       2.0,    # 权重之外的 KV/context/workspace 估算
    "ANCHOR_ALLOW_APPROX":   False,  # True 才允许用 Q8_0 之类做"近似参照"

    # ---------- 开关 ----------
    "FORCE_RERUN":      False,   # 最后手段：忽略指纹强制全跑
    "ALLOW_REQUANTIZE": False,   # BASE_KIND 是量化类型时必须显式打开（二次量化，仅 pilot）
    "HASH_GGUF":        True,    # 对产物算 sha256
    "BOOTSTRAP_N":      10000,
    "SEED":             1234,
}


def _apply_run_switches():
    """把 RUN_S2 这一个开关展开成 S2 需要的全部设置。

    RUN_S2=False 时**什么都不改** —— 这一点是刻意的：在 Kaggle 上点 "Run All"
    不应该因为导入 notebook 就悄悄联网下载。想下载必须显式把 RUN_S2 改成 True。
    """
    if not RUN_S2:
        return {"run_s2": False}
    STAGES["S2_BUILD"] = True
    CFG["S2_AUTO_FETCH_SOURCE"] = True
    CFG["LLAMA_COMMIT"] = S2_PINNED["commit"]
    CFG["CUDA_ARCH"] = "75"    # T4 = sm_75；不指定时上游会编九套架构
    CFG["CUDA_NO_VMM"] = True  # Kaggle T4 上 CUDA::cuda_driver target 找不到（v0.3.1）
    CFG["BUILD_CUDA"] = True   # 三个工具都有 CPU 路径，nvcc 缺失时预检会直接提示
    CFG["BUILD_JOBS"] = 4      # Kaggle 只有 4 核
    return {"run_s2": True,
            "stages": [k for k, v in STAGES.items() if v],
            "commit": CFG["LLAMA_COMMIT"],
            "cuda_arch": CFG["CUDA_ARCH"]}


RUN_SWITCHES = _apply_run_switches()

# 这条永远不允许改成 True。S9 会断言。
THIRD_PARTY_IN_COMPARISON = False

# ---- 派生路径 ----
ON_KAGGLE = Path("/kaggle/working").is_dir()
ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "_trlab_run"
WORK = ROOT / "trlab"
OUT  = WORK / "out"
LOGS = WORK / "logs"
CKPT = WORK / "ckpt"
MDLS = WORK / "models"
BUILD = WORK / "build-llama"      # P0-A：绝不在 /kaggle/input 下建 build
SRC_CACHE = WORK / "src"          # 自动下载的源码解压到这里
DL_DIR    = WORK / "dl"           # tarball 缓存（重跑不重下）
IN   = Path("/kaggle/input")

print(f"notebook      : v{NOTEBOOK_VERSION}")
print(f"运行位置      : {'Kaggle' if ON_KAGGLE else '本地/其他'}")
print(f"工作目录      : {WORK}")
print(f"llama.cpp build: {BUILD}")
print(f"RUN_S2        : {RUN_S2}"
      + ("（会下载 36.6 MiB 源码并编译；需要 Internet=On）" if RUN_S2
         else "（不联网、不下载）"))
print(f"开启的阶段    : {[k for k, v in STAGES.items() if v]}")
if RUN_S2:
    print(f"源码          : {S2_PINNED['url']}")
    print(f"                commit {S2_PINNED['commit']}  sha256 {S2_PINNED['sha256'][:16]}…")
print(f"量化 base     : {CFG['BASE_GGUF'] or '(未设置)'}  kind={CFG['BASE_KIND'] or '?'}")
print(f"imatrix base  : {CFG['IMATRIX_BASE'] or '(同量化 base)'}"
      f"  kind={CFG['IMATRIX_BASE_KIND'] or CFG['BASE_KIND'] or '?'}")
print(f"二次量化许可  : {'允许（仅 pilot，结果不可当高保真结论）' if CFG['ALLOW_REQUANTIZE'] else '禁止'}")
print(f"网络下载      : {'仅 S2 源码 tarball（RUN_S2=True）' if RUN_S2 else '禁止'}")
print(f"第三方量化入表: {'允许（不应发生）' if THIRD_PARTY_IN_COMPARISON else '禁止'}")


In [ ]:
import json, os, re, sys, time, math, hashlib, platform, shutil, struct, subprocess, tarfile, traceback
from datetime import datetime, timezone

for _d in (WORK, OUT, LOGS, CKPT, MDLS):
    _d.mkdir(parents=True, exist_ok=True)

T0 = time.time()
STATE_PATH = CKPT / "state.json"


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - T0:7.1f}s] {msg}", flush=True)


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
    tmp.replace(path)
    return path


def load_state():
    if STATE_PATH.exists():
        try:
            return json.loads(STATE_PATH.read_text(encoding="utf-8"))
        except Exception:
            log("state.json 损坏，重新开始")
    return {}


STATE = load_state()


def save_state():
    write_json(STATE_PATH, STATE)


def _get(stage):
    """取某阶段的 result，只有 status=='ok' 才算数。"""
    r = STATE.get(stage) or {}
    return r.get("result") if r.get("status") == "ok" else None


# ---------------- 异常 ----------------

class BudgetExceeded(RuntimeError):
    """墙钟 / 显存 / 磁盘超限。故意抛异常，而不是等 OOM killer。"""


class GuardError(RuntimeError):
    """前置条件不满足（路径缺失、二进制缺失、开关未打开）。"""


class CommandFailed(RuntimeError):
    def __init__(self, cmd, rc, tail, log_path):
        self.cmd, self.rc, self.tail, self.log_path = list(cmd), rc, tail, str(log_path)
        super().__init__(f"命令失败 rc={rc}: {' '.join(map(str, cmd))}\n--- tail ---\n{tail}")


class UnknownGGMLType(ValueError):
    """遇到不认识的 ggml type。拒绝按默认 8 bpw 假算。"""


# ---------------- 资源观测 ----------------

def vram_gb():
    try:
        import torch
        if not torch.cuda.is_available():
            return None
        free, total = torch.cuda.mem_get_info()
        return {"free": free / 2**30, "total": total / 2**30, "used": (total - free) / 2**30}
    except Exception:
        return None


def disk_gb(p):
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    u = shutil.disk_usage(p)
    return {"total": u.total / 2**30, "used": u.used / 2**30, "free": u.free / 2**30}


def guard(tag):
    """阶段**入口**检查。

    注意语义边界：这里只能拦住"进入时已经不满足"的情况。
    llama.cpp 是外部进程，它在运行中自己分配显存，Python 侧无法阻止它 OOM。
    运行中的保护靠 STAGE_BUDGET_SEC 的墙钟超时 + run_cmd 的 kill。
    """
    el = time.time() - T0
    if el > CFG["MAX_WALL_SEC"]:
        raise BudgetExceeded(
            f"{tag}: 累计墙钟 {el/3600:.2f} h 超过上限 {CFG['MAX_WALL_SEC']/3600:.2f} h")
    v = vram_gb()
    if v and v["used"] > CFG["MAX_VRAM_GB"]:
        raise BudgetExceeded(
            f"{tag}: 进入时显存已占 {v['used']:.2f} GB，超过上限 {CFG['MAX_VRAM_GB']} GB")
    d = disk_gb(WORK)
    if d["free"] < CFG["MIN_FREE_DISK_GB"]:
        raise BudgetExceeded(
            f"{tag}: 磁盘剩余 {d['free']:.2f} GB 低于下限 {CFG['MIN_FREE_DISK_GB']} GB")


def check_deadline(deadline, tag):
    if time.time() > deadline:
        raise BudgetExceeded(f"{tag}: 阶段预算已用尽")


def _est_need_bytes(size_bytes, extra_gb=2.0):
    """权重字节 + extra_gb（KV/context/workspace）→ 总需求字节。

    P0-B：括号必须显式。`size + 1 << 30` 会被解析成 `(size+1) << 30`，
    在 8 GiB 输入上把需求夸大到 8.59e9 GiB。
    """
    return int(size_bytes) + int(round(float(extra_gb) * (1 << 30)))


def _anchor_decision(anchor_bytes, vram_total_gb, base_kind, allow_approx,
                     extra_gb=2.0, margin_gb=1.0):
    """纯函数：给定锚点大小与显存，判断能不能跑、跑出来的结论算什么效力。"""
    usable = max(0.0, float(vram_total_gb) - float(margin_gb))
    required = _est_need_bytes(anchor_bytes, extra_gb) / 2**30
    tier = ANCHOR_TIERS.get(str(base_kind).lower(), "unknown")

    d = {
        "anchor_bytes": int(anchor_bytes),
        "anchor_gb": round(anchor_bytes / 2**30, 3),
        "required_gb": round(required, 3),
        "vram_total_gb": round(float(vram_total_gb), 2),
        "vram_usable_gb": round(usable, 2),
        "fits_on_gpu": required < usable,
        "tier": tier,
        "base_kind": str(base_kind),
    }
    if not d["fits_on_gpu"]:
        d.update(status="skipped",
                 reason=(f"需要 {required:.2f} GiB（权重 {anchor_bytes/2**30:.2f} + "
                         f"余量 {extra_gb} GiB），可用 {usable:.2f} GiB。"
                         "单张 T4 16GB 装不下 8B 高精度锚点。"))
    elif tier != "high" and not allow_approx:
        d.update(status="skipped",
                 reason=(f"锚点精度 = {base_kind}（tier={tier}），属二次量化近似，"
                         "不能用于『相对原始 checkpoint BF16』的有效性结论。"
                         "要跑近似参照请显式设 CFG['ANCHOR_ALLOW_APPROX']=True。"))
    elif tier != "high":
        d.update(status="ready", validity_scope="approx_only",
                 caveat="近似参照：可用来估算 rho 的量级，不可作为有效性证据")
    else:
        d.update(status="ready", validity_scope="high_precision")
    return d


# bf16/f16 才算高精度锚点；Q8 及以下都是二次量化
ANCHOR_TIERS = {
    "bf16": "high", "f16": "high", "fp16": "high",
    "q8_0": "approx", "q6_k": "approx", "q5_k": "approx",
    "q4_k_m": "approx", "q4_k": "approx", "q4_0": "approx",
}


def _bench_summary(tb):
    """把 S1 的 torch 微基准压成 S9 能安全渲染的字段。

    P0-C：绝不假设 bf16 一定拿到了 TFLOP/s。T4（CC 7.5）上 bf16 GEMM 可能直接报错。
    """
    tb = tb or {}
    fp = tb.get("fp16") or {}
    bf = tb.get("bf16") or {}
    cp = tb.get("copy") or {}
    out = {
        "torch_error": tb.get("error"),
        "cuda": tb.get("cuda"),
        "fp16_tflops": fp.get("tflops"),
        "bf16_tflops": bf.get("tflops"),
        "bf16_error": bf.get("error"),
        "bf16_available": bf.get("tflops") is not None,
        "fp16_available": fp.get("tflops") is not None,
        "ratio_bf16_over_fp16": None,
        "copy_gb_per_s": cp.get("gb_per_s_derived"),
        "copy_error": cp.get("error"),
    }
    if out["fp16_tflops"] and out["bf16_tflops"]:
        out["ratio_bf16_over_fp16"] = round(out["bf16_tflops"] / out["fp16_tflops"], 3)
    return out


# ---------------- 量化 base 的两条路径（v0.2.1） ----------------
# BASE_GGUF      = 量化 base（正式路径必须 f16/bf16）
# IMATRIX_BASE   = 算 imatrix 用的 base（留空 = BASE_GGUF；T4 上可指向 Q8_0）

# 已量化类型：从这些 base 再量化需要 --allow-requantize
QUANTIZED_KINDS = {
    "q8_0", "q8_1", "q6_k", "q5_k", "q5_0", "q5_1", "q4_k", "q4_k_m",
    "q4_0", "q4_1", "q3_k", "q2_k", "iq1_s", "iq1_m", "iq2_xxs", "iq2_xs",
    "iq2_s", "iq3_xxs", "iq3_s", "iq4_nl", "iq4_xs", "tq1_0", "tq2_0", "mxfp4",
}


def _base_requant_plan(base_kind, allow_requantize):
    """从给定 base 量化到 IQ2_XS/IQ4_XS 需要什么，结果算什么效力。

    上游事实（src/llama-quant.cpp 的 `llama_model_quantize_impl` 主循环）：
        if (ggml_is_quantized(tensor->type) && !params->allow_requantize)
            throw std::runtime_error("requantizing from type Q8_0 is disabled");
    `allow_requantize` 默认 false。tools/quantize/quantize.cpp 的帮助文本原文：
        "--allow-requantize   allow requantizing tensors that have already been quantized
         WARNING: this can severely reduce quality compared to quantizing from 16bit or 32bit!"
    """
    k = str(base_kind or "").strip().lower()
    is_q = k in QUANTIZED_KINDS
    d = {"base_kind": k, "base_is_quantized": is_q,
         "needs_allow_requantize": is_q, "allow_requantize": bool(allow_requantize),
         "flags": [], "validity_scope": None, "provenance_tag": None}

    if not k:
        d.update(status="blocked",
                 reason="CFG['BASE_KIND'] 未设置。必须写实际类型（f16 / bf16 / q8_0 …），"
                        "否则无法判断要不要 --allow-requantize。")
        return d
    if not is_q:
        d.update(status="ok", validity_scope="from_high_precision",
                 provenance_tag="primary")
        return d
    if not allow_requantize:
        d.update(status="blocked", reason=(
            f"BASE_KIND={k} 已经是量化类型。llama-quantize 遇到已量化的张量会直接抛 "
            f"`requantizing from type ... is disabled`，除非显式传 --allow-requantize；"
            f"而官方帮助对那个开关的原文是 "
            f"\"WARNING: this can severely reduce quality compared to quantizing from "
            f"16bit or 32bit!\"。正确做法：BASE_GGUF 指向同一 checkpoint 的 f16/bf16"
            f"（llama-quantize 是 CPU 路径，不吃显存），用 IMATRIX_BASE 指向这个 {k} 去算 imatrix。"
            f"只做 smoke 时，显式设 CFG['ALLOW_REQUANTIZE']=True，"
            f"产物会被标 requantized_pilot_only。"))
        return d
    d.update(status="ok", flags=["--allow-requantize"],
             validity_scope="requantized_pilot_only",
             provenance_tag="requantized_pilot_only",
             caveat=("二次量化：IQ2/IQ4 之间仍然同源可比，但不能作为"
                     "『相对原始 checkpoint BF16』的高保真低比特结论。"))
    return d


def _quant_cmd(qbin, imatrix, base, out_path, fmt, threads, plan):
    """llama-quantize 的 argv。**选项必须全部在位置参数之前。**

    上游 tools/quantize/quantize.cpp 的参数循环是
        for (; arg_idx < argc && strncmp(argv[arg_idx], "--", 2) == 0; arg_idx++)
    遇到第一个不以 `--` 开头的 token 就退出，之后一律当位置参数读
    （model / output / ftype / nthreads）。所以把 --allow-requantize 追加在末尾
    等于没传：它会被当成多余的位置参数丢掉，requantize 仍然被拒绝。
    """
    flags = [str(f) for f in ((plan or {}).get("flags") or [])]
    return ([str(qbin)] + flags
            + ["--imatrix", str(imatrix),
               str(base), str(out_path), str(fmt), str(threads)])


def _quant_fp(base_fp, imatrix_sha256, llama_commit, fmt, plan, params):
    """量化产物指纹：决定字节的东西全在里，notebook 版本只记录不参与哈希。

    只有指纹完全一致时才允许复用旧 GGUF —— 否则「换了 base 或 imatrix，
    S4 重跑了，但文件还是旧的」这种误报会一路传到 S6 的配对结论里。
    """
    payload = {
        "schema": 1,
        "base": base_fp,
        "imatrix_sha256": imatrix_sha256,
        "llama_commit": llama_commit,
        "fmt": str(fmt),
        "requant": (plan or {}).get("provenance_tag"),
        "params": params,
    }
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:32]


def _read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return None


def _retire_product(dst, tag=None):
    """指纹不符的旧产物：改名成 .stale-<tag>，并删掉更早的同类 stale。

    不直接覆盖 —— 新量化失败时旧文件还能对照；也不无限堆积，
    每种格式最多留一份 stale（两个 GiB 级文件不会把 working 撑爆）。
    """
    dst = Path(dst)
    if not dst.exists():
        return None
    if dst.stat().st_size == 0:
        dst.unlink()
        return None
    for old in sorted(dst.parent.glob(dst.name + ".stale-*")):
        old.unlink()
    stale = dst.with_name(f"{dst.name}.stale-{(tag or 'old')[:8]}")
    dst.rename(stale)
    return stale


def _base_consistency(scan_a, scan_b):
    """两份 base 是否来自同一 checkpoint：架构 + 张量名集合 + 每个张量形状 + 参数量。"""
    if not scan_a or not scan_b:
        return {"ok": None, "why": "有一侧没有 scan 结果"}
    if scan_a.get("architecture") != scan_b.get("architecture"):
        return {"ok": False,
                "why": f"architecture 不同：{scan_a.get('architecture')} "
                       f"vs {scan_b.get('architecture')}"}
    a = {t["name"]: tuple(t["dims"]) for t in scan_a.get("tensors", [])}
    b = {t["name"]: tuple(t["dims"]) for t in scan_b.get("tensors", [])}
    if set(a) != set(b):
        return {"ok": False,
                "why": f"张量名集合不同（只在 A: {sorted(set(a) - set(b))[:3]}；"
                       f"只在 B: {sorted(set(b) - set(a))[:3]}）"}
    bad = [n for n in a if a[n] != b[n]]
    if bad:
        n = bad[0]
        return {"ok": False,
                "why": f"{len(bad)} 个张量形状不同，例如 {n}: {a[n]} vs {b[n]}"}
    if scan_a.get("n_params") != scan_b.get("n_params"):
        return {"ok": False,
                "why": f"参数总数不同：{scan_a.get('n_params')} vs {scan_b.get('n_params')}"}
    return {"ok": True, "n_tensors": len(a), "architecture": scan_a.get("architecture"),
            "n_params": scan_a.get("n_params")}


def _rho_from_nll(anchor_nll, q4_nll, q2_nll):
    """rho = dNLL(Q2,Q4) / dNLL(Q4,anchor)，全部按「后减前」的正向定义。

    v0.2 写成 `anchor - Q4`，符号是反的：比值因为分子分母同号碰巧正确，
    但打印出来的 `dNLL(Q4,anchor)` 与代数定义相反。这里统一成 Q4 - anchor。
    """
    d = {"anchor_nll": anchor_nll, "q4_nll": q4_nll, "q2_nll": q2_nll}
    if anchor_nll is None or q4_nll is None or q2_nll is None:
        d.update(d4=None, d2=None, rho=None, reason="有 NLL 缺失")
        return d
    d4 = q4_nll - anchor_nll          # 锚点 → Q4 的退化（应为正）
    d2 = q2_nll - anchor_nll          # 锚点 → Q2 的退化（应比 d4 更大）
    d.update(d4=d4, d2=d2)
    if d4 <= 1e-9:
        d.update(rho=None,
                 reason=(f"dNLL(Q4,anchor)={d4:+.4f} ≤ 0：锚点没有比 Q4 更好，"
                         "rho 无定义（锚点无效，或 Q4 与锚点不可区分）"))
        return d
    d.update(rho=(d2 - d4) / d4, reason=None)
    return d


# ---------------- 路径解析 ----------------

def need(key):
    v = CFG.get(key, "")
    if not v:
        raise GuardError(f"CFG['{key}'] 未设置")
    p = Path(v)
    if not p.exists():
        raise GuardError(f"CFG['{key}'] 指向的路径不存在: {p}")
    return p


def _imatrix_base():
    """算 imatrix 用的 base。IMATRIX_BASE 留空则与量化 base 相同。"""
    if CFG["IMATRIX_BASE"]:
        p = Path(CFG["IMATRIX_BASE"])
        if not p.exists():
            raise GuardError(f"CFG['IMATRIX_BASE'] 指向的路径不存在: {p}")
        return p
    return need("BASE_GGUF")


def _imatrix_base_kind():
    return CFG["IMATRIX_BASE_KIND"] or CFG["BASE_KIND"]


def resolve_imatrix():
    """imatrix 的解析路径；不存在返回 None（不抛）。"""
    if CFG["IMATRIX_GGUF"]:
        p = Path(CFG["IMATRIX_GGUF"])
        return p if p.exists() else None
    p = MDLS / "imatrix.gguf"
    return p if p.exists() else None


def _manual_llama_src():
    """只找"用户手动提供"的源码：CFG['LLAMA_SRC'] 或 /kaggle/input 下的约定路径。

    与自动下载分开，是为了让 S2 能如实标注来源（manual / cache / downloaded），
    而不是把"上次自己下载的"误当成"用户挂载的"。
    """
    if CFG["LLAMA_SRC"]:
        p = Path(CFG["LLAMA_SRC"])
        return p if (p / "CMakeLists.txt").exists() else None
    for cand in [Path("/kaggle/input/llama.cpp"),
                 Path("/kaggle/input/llama-cpp/llama.cpp"),
                 Path("/kaggle/input/llamacpp/llama.cpp")]:
        if (cand / "CMakeLists.txt").exists():
            return cand
    return None


def _llama_src_path():
    """非抛版本，供指纹使用。手动路径优先，其次自动下载的缓存目录。"""
    p = _manual_llama_src()
    if p:
        return p
    q = SRC_CACHE / S2_PINNED["dir"]
    return q if (q / "CMakeLists.txt").exists() else None


def _model_path(fmt):
    """S4 产物的约定路径。"""
    return MDLS / f"qwen3-8b-{fmt}.gguf"


def llama_bin(name):
    """按 CFG -> WORK/build-llama -> 源码树 -> PATH 的顺序找可执行文件。

    P0-A：build 目录固定在 WORK 下，因为 /kaggle/input 是只读的。
    """
    if CFG["LLAMA_BIN"]:
        p = Path(CFG["LLAMA_BIN"]) / name
        if p.exists():
            return p
    for cand in (BUILD / "bin" / name, BUILD / name):
        if cand.exists():
            return cand
    src = _llama_src_path()
    if src:
        for cand in (src / "build" / "bin" / name, src / "build" / name):
            if cand.exists():
                return cand
    w = shutil.which(name)
    if w:
        return Path(w)
    raise GuardError(
        f"找不到 {name}。先跑 S2_BUILD（build 在 {BUILD}），或设置 CFG['LLAMA_BIN']。")


def _git_commit(p):
    try:
        r = subprocess.run(["git", "-C", str(p), "rev-parse", "HEAD"],
                           capture_output=True, text=True, timeout=30)
        return r.stdout.strip() or None
    except Exception:
        return None


def sha256_of(path, deadline=None, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
            if deadline and time.time() > deadline:
                return "aborted:deadline"
    return h.hexdigest()


# ---------------- 工具链预检（S2 用） ----------------

def _toolchain_report():
    """cmake / nvcc / g++ 的版本。不抛异常，拿不到就写"不可用"。"""
    out = {}
    for name, cmd in (("cmake", ["cmake", "--version"]),
                      ("nvcc", ["nvcc", "--version"]),
                      ("gxx", ["g++", "-dumpversion"]),
                      ("gcc", ["gcc", "-dumpversion"])):
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=20)
            txt = (r.stdout or r.stderr or "").strip()
            out[name] = (txt.split("\n")[0][:120] if r.returncode == 0
                         else f"rc={r.returncode}")
        except Exception as e:
            out[name] = f"不可用: {type(e).__name__}"
    return out


def _cuda_preflight(toolchain, build_cuda):
    """BUILD_CUDA=True 但 nvcc 不可用时提前报错，而不是让 cmake 在 FATAL_ERROR 里淹掉。"""
    nvcc = str((toolchain or {}).get("nvcc") or "")
    if not build_cuda:
        return {"ok": True,
                "note": "BUILD_CUDA=False：只构建 CPU 版；三个工具都有 CPU 路径"}
    if (not nvcc) or nvcc.startswith("不可用") or nvcc.startswith("rc="):
        return {"ok": False,
                "note": ("BUILD_CUDA=True 但 nvcc 不可用。改 CFG['BUILD_CUDA']=False —— "
                         "llama-imatrix / llama-quantize / llama-perplexity 都能走 CPU，"
                         "只是慢；或换一个带 CUDA 的镜像。")}
    return {"ok": True, "nvcc": nvcc}


# ---------------- 输入指纹（P1-D） ----------------

def _path_fp(p):
    """文件：size + mtime + 首尾各 1 MiB 哈希。目录：CMakeLists + git commit。"""
    if p is None:
        return None
    p = Path(p)
    if not p.exists():
        return None
    if p.is_dir():
        d = {"kind": "dir"}
        cml = p / "CMakeLists.txt"
        if cml.exists():
            st = cml.stat()
            d.update(cmake_size=st.st_size, cmake_mtime=st.st_mtime_ns)
        d["git"] = _git_commit(p)
        return d
    st = p.stat()
    rec = {"kind": "file", "size": st.st_size, "mtime": st.st_mtime_ns}
    try:
        with open(p, "rb") as f:
            rec["head"] = hashlib.sha256(f.read(1 << 20)).hexdigest()[:16]
            if st.st_size > (2 << 20):
                f.seek(-(1 << 20), os.SEEK_END)
                rec["tail"] = hashlib.sha256(f.read(1 << 20)).hexdigest()[:16]
    except OSError as e:
        rec["read_error"] = repr(e)
    return rec


# 每阶段依赖：CFG 键 + 文件解析器（失败即 None）
STAGE_DEPS = {
    "S0_ENV":         ([], []),
    "S1_SMOKE":       (["PARAM_COUNT"], []),
    "S2_BUILD":       (["LLAMA_SRC", "LLAMA_BIN", "LLAMA_COMMIT", "BUILD_CUDA",
                        "CUDA_ARCH", "CUDA_NO_VMM", "BUILD_JOBS"],
                       [lambda: _llama_src_path()]),
    "S3_IMATRIX":     (["BASE_GGUF", "BASE_KIND", "IMATRIX_BASE", "IMATRIX_BASE_KIND",
                        "CALIB_TXT", "IMATRIX_CTX", "IMATRIX_CHUNKS", "IMATRIX_GGUF"],
                       [lambda: _imatrix_base(), lambda: need("CALIB_TXT")]),
    "S4_QUANT":       (["BASE_GGUF", "BASE_KIND", "IMATRIX_BASE", "IMATRIX_BASE_KIND",
                        "IMATRIX_GGUF", "ALLOW_REQUANTIZE"],
                       [lambda: need("BASE_GGUF"), resolve_imatrix]),
    "S5_BYTES":       (["BASE_GGUF", "BASE_KIND"],
                       [lambda: need("BASE_GGUF"),
                        lambda: _model_path("IQ2_XS"), lambda: _model_path("IQ4_XS")]),
    "S6_NLL":         (["EVAL_TXT", "EVAL_SOURCE", "NLL_CTX", "NLL_CHUNKS_PER_GROUP",
                        "NLL_GROUPS", "SEED"],
                       [lambda: need("EVAL_TXT"),
                        lambda: _model_path("IQ2_XS"), lambda: _model_path("IQ4_XS")]),
    "S7_QUALITY":     (["QUALITY_PROMPTS", "QUALITY_N_PREDICT", "SEED"],
                       [lambda: _model_path("IQ2_XS"), lambda: _model_path("IQ4_XS")]),
    "S8_BF16_ANCHOR": (["BASE_GGUF", "BASE_KIND", "ANCHOR_EXTRA_GB", "ANCHOR_ALLOW_APPROX"],
                       [lambda: need("BASE_GGUF")]),
    "S9_REPORT":      ([], []),
    "SX_EXPLORE":     (["EXPLORE_GGUF"], [lambda: need("EXPLORE_GGUF")]),
}


def _fingerprint(name):
    """阶段输入指纹。任何依赖变化都会改变它，从而让旧结果失效。"""
    cfg_keys, file_res = STAGE_DEPS.get(name, ([], []))
    payload = {
        "nb": NOTEBOOK_VERSION,
        "stage": name,
        "cfg": {k: CFG.get(k) for k in cfg_keys},
        "files": {},
    }
    for i, r in enumerate(file_res):
        try:
            p = r()
        except Exception:
            p = None
        payload["files"][f"f{i}"] = _path_fp(p)
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:32]


# ---------------- 阶段执行器 ----------------

def run_stage(name, fn):
    """开关 + 指纹断点 + 预算 + JSON 落盘的统一入口。"""
    if not STAGES.get(name, False):
        log(f"[{name}] SKIP —— STAGES['{name}'] 为 False")
        return None

    fp = _fingerprint(name)
    prev = STATE.get(name) or {}
    always = name in ALWAYS_RERUN

    if always:
        log(f"[{name}] 强制重跑（在 ALWAYS_RERUN 里，汇总不能停留在旧结果）")
    elif prev.get("status") == "ok" and prev.get("fingerprint") == fp and not CFG["FORCE_RERUN"]:
        log(f"[{name}] RESUME —— 指纹一致 {fp}，产物 {prev.get('result_path', '')}")
        return prev.get("result")
    elif prev.get("status") == "ok" and not CFG["FORCE_RERUN"]:
        log(f"[{name}] INVALIDATE —— 输入指纹变化 "
            f"{prev.get('fingerprint')} → {fp}，重跑该阶段")

    budget = CFG["STAGE_BUDGET_SEC"].get(name, 600)
    log(f"[{name}] START（预算 {budget} s）")
    t = time.time()
    rec = {"stage": name,
           "started": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "fingerprint": fp, "notebook_version": NOTEBOOK_VERSION}
    try:
        guard(name)
        result = fn(t + budget)
        rec.update(status="ok", elapsed_sec=round(time.time() - t, 2),
                   result=result, result_path=str(OUT / f"{name}.json"))
        write_json(OUT / f"{name}.json", {"stage": name, "meta": rec, "result": result})
        log(f"[{name}] OK  {rec['elapsed_sec']} s  →  {rec['result_path']}")
    except Exception as e:
        rec.update(status="error", elapsed_sec=round(time.time() - t, 2),
                   error=f"{type(e).__name__}: {e}",
                   traceback=traceback.format_exc()[-2000:])
        write_json(OUT / f"{name}.json", {"stage": name, "meta": rec, "result": None})
        log(f"[{name}] FAIL  {rec['error']}")

    STATE[name] = rec
    save_state()
    return rec.get("result")


# ---------------- 外部命令 ----------------

def run_cmd(cmd, log_path, deadline, cwd=None, env=None):
    """跑外部命令。rc != 0 一律抛 CommandFailed（P1-E：不再静默忽略返回码）。"""
    cmd = [str(c) for c in cmd]
    log(f"  $ {' '.join(cmd)}")
    t = time.time()
    with open(log_path, "wb") as lf:
        p = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, cwd=cwd, env=env)
        while p.poll() is None:
            if time.time() > deadline:
                p.kill()
                raise BudgetExceeded(f"命令超预算被杀: {cmd[0]}")
            time.sleep(0.5)
    rc = p.returncode
    tail = Path(log_path).read_text(errors="replace")[-1800:]
    log(f"  rc={rc}  {time.time() - t:.1f} s  →  {log_path}")
    if rc != 0:
        raise CommandFailed(cmd, rc, tail, log_path)
    return rc, tail


def atomic_replace(src, dst):
    """先写 .partial，验完再原子改名。避免把截断/旧文件当成功产物。"""
    src, dst = Path(src), Path(dst)
    if not src.exists():
        raise GuardError(f"atomic_replace: 源文件不存在 {src}")
    os.replace(src, dst)
    return dst


def partial_path(dst):
    dst = Path(dst)
    return dst.with_name(dst.name + ".partial")


# ---------------- 统计 ----------------

def paired_bootstrap(diffs, n=None, seed=None):
    """配对差值的 bootstrap 均值 CI。n<2 时不给 CI，而不是崩。"""
    import random
    n = n or CFG["BOOTSTRAP_N"]
    seed = CFG["SEED"] if seed is None else seed
    m = len(diffs)
    if m == 0:
        return {"n": 0, "mean": None, "ci95": None}
    mean = sum(diffs) / m
    if m < 2:
        return {"n": m, "mean": mean, "ci95": None, "note": "只有 1 对，无法给 CI"}
    rng = random.Random(seed)
    means = []
    for _ in range(n):
        s = 0.0
        for _ in range(m):
            s += diffs[rng.randrange(m)]
        means.append(s / m)
    means.sort()
    return {"n": m, "mean": mean,
            "ci95": [means[int(0.025 * n)], means[int(0.975 * n)]],
            "boot": n}


log(f"运行时已就绪  notebook v{NOTEBOOK_VERSION}")
log(f"断点状态: {list(STATE.keys()) if STATE else '空'}")


## S0 · 环境 / GPU / 磁盘检查

这一步不加载任何模型，只回答「这台机器能干什么」。

In [ ]:
def stage_s0_env(deadline):
    info = {}
    info["notebook_version"] = NOTEBOOK_VERSION
    info["where"] = "kaggle" if ON_KAGGLE else "local"
    info["utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    info["python"] = sys.version.split()[0]
    info["platform"] = platform.platform()
    info["cpu_count"] = os.cpu_count()

    # ---- RAM ----
    try:
        import psutil
        vm = psutil.virtual_memory()
        ram = {"total_gb": round(vm.total / 2**30, 2), "available_gb": round(vm.available / 2**30, 2)}
    except Exception:
        try:
            ram = {"total_gb": round(os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 2**30, 2),
                   "available_gb": None}
        except Exception as e:
            ram = {"error": repr(e)}
    info["ram"] = ram

    # ---- GPU ----
    gpus = []
    try:
        import torch
        info["torch"] = torch.__version__
        info["torch_cuda"] = torch.version.cuda
        info["cuda_available"] = bool(torch.cuda.is_available())
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            free, total = torch.cuda.mem_get_info(i)
            gpus.append({
                "idx": i, "name": p.name, "cc": f"{p.major}.{p.minor}",
                "total_gb": round(total / 2**30, 2),
                "free_gb": round(free / 2**30, 2),
                "sm_count": p.multi_processor_count,
                "native_bf16": p.major >= 8,   # CC<8.0 没有原生 bf16（T4 = 7.5）
            })
    except Exception as e:
        info["torch_error"] = repr(e)
    info["gpus"] = gpus
    info["vram_usable_gb_est"] = (round(max(g["total_gb"] for g in gpus) - 1.0, 2)
                                  if gpus else 0.0)

    try:
        info["nvidia_smi"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:
        info["nvidia_smi"] = f"(unavailable: {e})"

    # ---- 磁盘 ----
    disks = {}
    for p in ["/kaggle/working", "/kaggle/temp", "/kaggle/input", str(WORK)]:
        try:
            if Path(p).exists():
                disks[p] = {k: round(v, 2) for k, v in disk_gb(p).items()}
        except Exception as e:
            disks[p] = {"error": repr(e)}
    info["disks_gb"] = disks

    # ---- 工具链 ----
    tools = {}
    for b in ["cmake", "g++", "gcc", "nvcc", "git", "make",
              "llama-imatrix", "llama-quantize", "llama-perplexity", "llama-cli"]:
        tools[b] = shutil.which(b)
    info["tools"] = tools

    # ---- Python 包 ----
    pkgs = {}
    for name in ["torch", "numpy", "psutil", "gguf", "transformers", "datasets"]:
        try:
            import importlib.metadata as md
            pkgs[name] = md.version(name)
        except Exception:
            pkgs[name] = None
    info["packages"] = pkgs

    # ---- /kaggle/input 清单（这是你填 CFG 路径的依据）----
    inv = []
    if IN.is_dir():
        for d in sorted(IN.iterdir()):
            if not d.is_dir():
                continue
            n_files, tot, ggufs = 0, 0, []
            try:
                for j, p in enumerate(d.rglob("*")):
                    if j % 200 == 0:
                        check_deadline(deadline, "S0_ENV 扫描 /kaggle/input")
                    if p.is_file():
                        n_files += 1
                        try:
                            sz = p.stat().st_size
                        except OSError:
                            continue
                        tot += sz
                        if p.suffix.lower() == ".gguf":
                            ggufs.append({"path": str(p), "gb": round(sz / 2**30, 3)})
            except Exception as e:
                inv.append({"dataset": d.name, "error": repr(e)})
                continue
            inv.append({"dataset": d.name, "files": n_files,
                        "gb": round(tot / 2**30, 2), "gguf": ggufs[:20],
                        "gguf_count": len(ggufs)})
    info["input_datasets"] = inv

    # ---- 结论性提示 ----
    warn = []
    if not gpus:
        warn.append("没有可用 GPU。S6/S7 的 llama.cpp 会退到 CPU，速度慢几十倍。")
    elif any(not g["native_bf16"] for g in gpus):
        warn.append("GPU 不支持原生 bf16（CC<8.0）。高精度锚点只能用 f16；"
                    "bf16 矩阵乘可能直接报错 —— S1 会实测，S9 不会假设它可用。")
    if info["vram_usable_gb_est"] and info["vram_usable_gb_est"] < 17.3:
        warn.append(f"可用显存约 {info['vram_usable_gb_est']:.1f} GiB，"
                    "装不下 8B f16（15.25 GiB 权重 + KV）。S8 会拒绝执行。")
    if not any(x.get("gguf_count") for x in inv if isinstance(x, dict)):
        warn.append("/kaggle/input 下没有找到 .gguf。S3-S7 需要你手动挂载 base GGUF。")
    if tools["cmake"] is None:
        warn.append("没有 cmake，S2_BUILD 无法执行。")
    info["warnings"] = warn

    # ---- 打印 ----
    print("=" * 78)
    print(f"Python {info['python']} | CPU {info['cpu_count']} | RAM {ram}")
    for g in gpus:
        print(f"GPU{g['idx']}: {g['name']}  CC{g['cc']}  {g['total_gb']} GiB "
              f"(free {g['free_gb']})  SM={g['sm_count']}  native_bf16={g['native_bf16']}")
    print(f"可用显存估计: {info['vram_usable_gb_est']} GiB")
    print("-" * 78)
    for k, v in disks.items():
        print(f"disk {k:22s} total={v.get('total')} GiB  free={v.get('free')} GiB")
    print("-" * 78)
    print("tools:", {k: (v.split("/")[-1] if v else None) for k, v in tools.items()})
    print("pkgs :", pkgs)
    print("-" * 78)
    if inv:
        print("/kaggle/input:")
        for d in inv:
            if "error" in d:
                print(f"  {d['dataset']}: ERROR {d['error']}")
            else:
                print(f"  {d['dataset']}: {d['files']} files, {d['gb']} GiB, {d['gguf_count']} gguf")
                for g in d["gguf"]:
                    print(f"      {g['gb']:8.2f} GiB  {g['path']}")
    else:
        print("/kaggle/input 为空")
    print("=" * 78)
    for w in warn:
        print("!! " + w)

    return info


s0 = run_stage("S0_ENV", stage_s0_env)


## 关于 GGUF 字节统计

`S5_BYTES` 用**纯 Python 的 GGUF 头解析器**（约 110 行，无第三方依赖）逐张量读 `type` / `dims` / `offset`，按 ggml 的 block 结构算真实字节：

```
bytes(tensor) = ceil(n_elements / block_size) * type_size
```

| 类型 | block | bytes | bpw |
|---|---|---|---|
| `F16` / `BF16` | 1 | 2 | 16.0 |
| `Q4_K` | 256 | 144 | 4.5 |
| `Q6_K` | 256 | 210 | 6.5625 |
| `Q8_0` | 32 | 34 | 8.5 |
| `IQ2_XS` | 256 | 74 | **2.3125** |
| `IQ4_XS` | 256 | 136 | **4.25** |
| `IQ2_XXS` / `TQ2_0` | 256 | 66 | 2.0625 |
| `MXFP4` | 32 | 17 | 4.25 |

**v0.2 起未知 type 会 fail closed。** 旧版 `GGML_TYPES.get(tid, ("UNKNOWN", 1, 1))` 把不认识的类型当成"1 个元素 1 字节"，也就是 **8 bpw**，然后正常写进表里 —— 这会把一个错误的 bpw 伪装成有效结果。现在抛 `UnknownGGMLType`，S5 记 `byte_breakdown_available: False`，不出数。

S1 会写一个**合成 GGUF** 再读回来，逐项断言字节数，这样解析器在碰大文件之前就被验证过。


In [ ]:
# ---------------- 纯 Python GGUF 头解析（无第三方依赖） ----------------
# (name, block_size, type_size_bytes)，来自 ggml-common.h 的结构体实际大小
GGML_TYPES = {
    0:  ("F32",      1,   4),
    1:  ("F16",      1,   2),
    2:  ("Q4_0",     32,  18),
    3:  ("Q4_1",     32,  20),
    6:  ("Q5_0",     32,  22),
    7:  ("Q5_1",     32,  24),
    8:  ("Q8_0",     32,  34),
    9:  ("Q8_1",     32,  36),
    10: ("Q2_K",     256, 84),
    11: ("Q3_K",     256, 110),
    12: ("Q4_K",     256, 144),
    13: ("Q5_K",     256, 176),
    14: ("Q6_K",     256, 210),
    15: ("Q8_K",     256, 292),
    16: ("IQ2_XXS",  256, 66),
    17: ("IQ2_XS",   256, 74),
    18: ("IQ3_XXS",  256, 98),
    19: ("IQ1_S",    256, 50),
    20: ("IQ4_NL",   32,  18),
    21: ("IQ3_S",    256, 110),
    22: ("IQ2_S",    256, 82),
    23: ("IQ4_XS",   256, 136),
    24: ("I8",       1,   1),
    25: ("I16",      1,   2),
    26: ("I32",      1,   4),
    27: ("I64",      1,   8),
    28: ("F64",      1,   8),
    29: ("IQ1_M",    256, 56),
    30: ("BF16",     1,   2),
    34: ("TQ1_0",    256, 54),
    35: ("TQ2_0",    256, 66),
    36: ("MXFP4",    32,  17),
}

# 标量 value_type -> 字节数（GGUF 规范）
_GGUF_SCALAR = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}


def ggml_type_info(tid, strict=True):
    """P1-G：未知 type 必须 fail closed，不能按 8 bpw 假算。"""
    info = GGML_TYPES.get(tid)
    if info is None:
        if strict:
            raise UnknownGGMLType(
                f"未知 ggml type_id={tid}。拒绝按默认尺寸假算字节 —— "
                "该文件的字节拆分不可用。请升级 GGML_TYPES 表，或跳过此文件。")
        return ("UNKNOWN", None, None)
    return info


class _Reader:
    def __init__(self, f):
        self.f = f

    def read(self, n):
        b = self.f.read(n)
        if len(b) != n:
            raise EOFError(f"unexpected EOF: want {n}, got {len(b)}")
        return b

    def u32(self):
        return struct.unpack("<I", self.read(4))[0]

    def u64(self):
        return struct.unpack("<Q", self.read(8))[0]

    def string(self):
        n = self.u64()
        return self.read(n).decode("utf-8", "replace")


def _skip_value(r, vt):
    """跳过任意 metadata value，不物化大数组（vocab 有 15 万个字符串）。"""
    if vt == 8:
        n = r.u64()
        r.f.seek(n, 1)
    elif vt == 9:
        et = r.u32()
        cnt = r.u64()
        if et in _GGUF_SCALAR:
            r.f.seek(_GGUF_SCALAR[et] * cnt, 1)
        elif et == 8:
            for _ in range(cnt):
                n = r.u64()
                r.f.seek(n, 1)
        else:
            raise ValueError(f"不支持的数组元素类型 {et}")
    elif vt in _GGUF_SCALAR:
        r.read(_GGUF_SCALAR[vt])
    else:
        raise ValueError(f"不支持的 value type {vt}")


def gguf_scan(path, deadline=None, strict_types=True):
    """读 GGUF 头，返回逐张量的类型/形状/真实字节。"""
    path = Path(path)
    with open(path, "rb") as f:
        r = _Reader(f)
        magic = r.read(4)
        if magic != b"GGUF":
            raise ValueError(f"不是 GGUF 文件（magic={magic!r}）: {path}")
        ver = r.u32()
        if ver not in (2, 3):
            raise ValueError(f"不支持的 GGUF 版本 {ver}")
        n_tensors = r.u64()
        n_kv = r.u64()

        WANT = {"general.architecture", "general.name", "general.file_type"}
        kv = {}
        for _ in range(n_kv):
            k = r.string()
            vt = r.u32()
            if k == "general.alignment" and vt == 4:
                kv[k] = r.u32()
            elif k in WANT and vt == 8:
                kv[k] = r.string()
            else:
                _skip_value(r, vt)

        align = int(kv.get("general.alignment", 32))
        tensors = []
        for i in range(n_tensors):
            if deadline and i % 64 == 0:
                check_deadline(deadline, "gguf_scan")
            name = r.string()
            nd = r.u32()
            dims = [r.u64() for _ in range(nd)]
            tt = r.u32()
            off = r.u64()
            tensors.append({"name": name, "dims": dims, "type_id": tt, "offset": off})
        data_start = f.tell()

    data_start += (-data_start) % align

    per_type, per_class = {}, {}
    n_params, tensor_bytes = 0, 0
    for t in tensors:
        n = 1
        for d in t["dims"]:
            n *= d
        t["n_elements"] = n
        n_params += n

        tname, blk, tsz = ggml_type_info(t["type_id"], strict=strict_types)
        t["type"] = tname
        if blk is None:
            t["nbytes"] = None
            per_type[tname] = per_type.get(tname, 0) + 0
            continue
        nb = ((n + blk - 1) // blk) * tsz
        t["nbytes"] = nb
        tensor_bytes += nb
        per_type[tname] = per_type.get(tname, 0) + nb

        nm = t["name"].lower()
        if "token_embd" in nm or "output.weight" in nm:
            cls = "embed/head"
        elif "attn_" in nm:
            cls = "attn"
        elif "ffn_" in nm:
            cls = "ffn"
        else:
            cls = "other"
        per_class[cls] = per_class.get(cls, 0) + nb

    fsize = path.stat().st_size
    unknown_types = sorted({t["type"] for t in tensors if t["nbytes"] is None})
    return {
        "path": str(path),
        "file_bytes": fsize,
        "file_gb": round(fsize / 2**30, 3),
        "gguf_version": ver,
        "architecture": kv.get("general.architecture"),
        "alignment": align,
        "n_tensors": len(tensors),
        "n_params": n_params,
        "tensor_bytes": tensor_bytes,
        "header_bytes": fsize - tensor_bytes,
        # 有未知 type 时 tensor_bytes 只是下界，此时 bpw 是假数，必须给 None
        "bpw_effective": (round(tensor_bytes * 8 / n_params, 4)
                          if (n_params and not unknown_types) else None),
        "bpw_filesize": round(fsize * 8 / n_params, 4) if n_params else None,
        "bytes_per_type": dict(sorted(per_type.items(), key=lambda x: -x[1])),
        "bytes_per_class": dict(sorted(per_class.items(), key=lambda x: -x[1])),
        "unknown_types": unknown_types,
        "tensors": tensors,
    }


print("GGUF 解析器已加载（未知 type 会 fail closed）")


In [ ]:
# ---------------- 跨阶段共享的评测工具 ----------------
# 故意放在独立的 cell（而不是某个阶段里）：这样即使你跳过中间的 cell
# 单独跑后面的阶段，也不会因为函数没定义而 NameError。

PPL_RE = re.compile(r"final estimate:\s*PPL\s*=\s*([0-9.]+)\s*\+/-\s*([0-9.]+)", re.I)


def _fit_ngl(nbytes, vram_usable_gb, margin_gb=1.0):
    """权重能不能进显存。返回 llama.cpp 的 -ngl 值。"""
    if not vram_usable_gb:
        return 0
    need = nbytes / 2**30 + margin_gb
    return 99 if need < vram_usable_gb else 0


def split_eval(path, n_groups, out_dir):
    """把评测文本切成**恰好** n_groups 份不相交的连续子文件。

    P1-F：旧版用 ceil(len/n)，41 段 8 组时只会得到 7 组。
    现在整除 + 余数分配，组数严格等于 min(n_groups, 段数)，且总段数不变。
    """
    txt = Path(path).read_text(errors="replace")
    paras = [p.strip() for p in re.split(r"\n\s*\n", txt) if p.strip()]
    if len(paras) < n_groups:
        paras = [p for p in txt.split("\n") if p.strip()]
    if not paras:
        raise ValueError(f"评测语料为空: {path}")

    n = max(1, min(int(n_groups), len(paras)))
    base, rem = divmod(len(paras), n)
    sizes = [base + (1 if i < rem else 0) for i in range(n)]
    assert sum(sizes) == len(paras) and len(sizes) == n

    groups, i = [], 0
    for s in sizes:
        groups.append(paras[i:i + s])
        i += s

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths, meta = [], []
    for gi, g in enumerate(groups):
        fp = out_dir / f"eval_g{gi:03d}.txt"
        body = "\n\n".join(g)
        fp.write_text(body, encoding="utf-8")
        paths.append(fp)
        meta.append({"group": gi, "paras": len(g), "chars": len(body)})
    return paths, meta


def run_ppl(model_path, text_path, deadline, tag):
    """跑 llama-perplexity，解析 Final estimate。返回 NLL = ln(PPL)。

    run_cmd 已经在 rc != 0 时抛 CommandFailed，所以能走到解析这一步
    说明进程正常退出（P1-E）。
    """
    v = vram_gb()
    usable = (v["total"] - 1.0) if v else 0.0
    ngl = _fit_ngl(Path(model_path).stat().st_size, usable)
    cmd = [llama_bin("llama-perplexity"),
           "-m", model_path, "-f", text_path,
           "-c", CFG["NLL_CTX"],
           "--chunks", CFG["NLL_CHUNKS_PER_GROUP"],
           "-ngl", ngl, "-t", os.cpu_count() or 4,
           "-b", "512", "--seed", CFG["SEED"]]
    logf = LOGS / f"ppl_{tag}.log"
    run_cmd(cmd, logf, deadline)

    full = logf.read_text(errors="replace")
    m = PPL_RE.search(full)
    if not m:
        raise RuntimeError(f"{tag}: rc=0 但解析不到 PPL。日志尾部:\n{full[-1200:]}")
    ppl = float(m.group(1))
    return {"ppl": ppl, "ppl_err": float(m.group(2)), "nll": math.log(ppl),
            "ngl": ngl, "chunks": CFG["NLL_CHUNKS_PER_GROUP"], "ctx": CFG["NLL_CTX"]}


print("共享评测工具已加载：_fit_ngl / split_eval / run_ppl")


## S1 · 短时 smoke test

先证明解析器、算术和统计工具是对的，再碰大文件。这里同时是**回归闸门**：五条断言覆盖 v0.1 的四个缺陷。

In [ ]:
def gguf_write_synth(path, specs, alignment=32, strict_types=True):
    """写一个最小但合法的 GGUF v3，用于自检解析器。specs = [(name, dims, type_id)]

    strict_types=False 只给「故意构造未知 type 的坏文件」用：
    未知 type 没有块尺寸，这里塞占位字节让文件长度成立，
    扫描端仍会（也应该）拒绝用它算字节。
    """
    import io
    buf = io.BytesIO()

    def u32(x): buf.write(struct.pack("<I", x))
    def u64(x): buf.write(struct.pack("<Q", x))

    def s(x):
        b = x.encode()
        u64(len(b))
        buf.write(b)

    buf.write(b"GGUF")
    u32(3)
    u64(len(specs))
    u64(2)
    s("general.architecture"); u32(8); s("llama")
    s("general.alignment");    u32(4); u32(alignment)

    sizes, cur = [], 0
    for name, dims, tid in specs:
        s(name)
        u32(len(dims))
        for d in dims:
            u64(d)
        u32(tid)
        u64(cur)
        n = 1
        for d in dims:
            n *= d
        _, blk, tsz = ggml_type_info(tid, strict=strict_types)
        if blk is None:
            # 未知 type：占位尺寸，只为了把文件写出来。scan 端不会采用它。
            blk, tsz = 256, 64
        nb = ((n + blk - 1) // blk) * tsz
        sizes.append(nb)
        cur += ((nb + alignment - 1) // alignment) * alignment

    while len(buf.getvalue()) % alignment:
        buf.write(b"\x00")
    for nb in sizes:
        buf.write(b"\x00" * nb)

    Path(path).write_bytes(buf.getvalue())
    return Path(path)


def _torch_microbench(deadline):
    """fp16 vs bf16 矩阵乘 + 拷贝带宽。

    T4（CC 7.5）上 bf16 GEMM 可能直接报错，这里只如实记录，
    不做任何"应该能拿到 TFLOP/s"的假设。
    """
    out = {}
    try:
        import torch
    except Exception as e:
        return {"error": f"torch 不可用: {e}"}
    if not torch.cuda.is_available():
        return {"cuda": False, "note": "无 GPU，跳过"}

    N = 2048
    flops = 2.0 * N**3
    for dt, label in ((torch.float16, "fp16"), (torch.bfloat16, "bf16")):
        try:
            check_deadline(deadline, "S1 矩阵乘")
            a = torch.randn(N, N, device="cuda", dtype=dt)
            b = torch.randn(N, N, device="cuda", dtype=dt)
            for _ in range(2):
                c = a @ b
            torch.cuda.synchronize()
            t = time.time()
            for _ in range(5):
                c = a @ b
            torch.cuda.synchronize()
            el = (time.time() - t) / 5
            out[label] = {"ms": round(el * 1000, 3), "tflops": round(flops / el / 1e12, 2)}
            del a, b, c
            torch.cuda.empty_cache()
        except Exception as e:
            out[label] = {"error": f"{type(e).__name__}: {e}"}
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

    try:
        nbytes = 256 << 20
        x = torch.empty(nbytes // 4, device="cuda", dtype=torch.float32)
        y = torch.empty_like(x)
        for _ in range(2):
            y.copy_(x)
        torch.cuda.synchronize()
        t = time.time()
        for _ in range(5):
            y.copy_(x)
        torch.cuda.synchronize()
        el = (time.time() - t) / 5
        out["copy"] = {"gb_moved": round(2 * nbytes / 2**30, 3),
                       "ms": round(el * 1000, 2),
                       "gb_per_s_derived": round(2 * nbytes / el / 1e9, 1),
                       "note": "read+write 的派生值，不是 DRAM 计数器实测"}
        del x, y
        torch.cuda.empty_cache()
    except Exception as e:
        out["copy"] = {"error": f"{type(e).__name__}: {e}"}

    try:
        free, total = torch.cuda.mem_get_info()
        out["vram_after_gb"] = {"free": round(free / 2**30, 2), "total": round(total / 2**30, 2)}
    except Exception:
        pass
    return out


def _disk_bench(deadline, mb=256):
    p = WORK / "smoke_disk.bin"
    chunk = b"\x00" * (1 << 20)
    t = time.time()
    try:
        with open(p, "wb") as f:
            for _ in range(mb):
                f.write(chunk)
            f.flush()
            os.fsync(f.fileno())
        el = time.time() - t
        return {"mb": mb, "sec": round(el, 3), "mb_per_s": round(mb / el, 1)}
    finally:
        if p.exists():
            p.unlink()


def stage_s1_smoke(deadline):
    res = {"checks": {}, "bench": {}}

    # ---- 1. 合成 GGUF 往返 ----
    specs = [
        ("token_embd.weight",     (256, 128), 1),   # F16
        ("blk.0.ffn_gate.weight", (256, 256), 17),  # IQ2_XS
        ("blk.0.ffn_up.weight",   (256, 256), 23),  # IQ4_XS
        ("blk.0.attn_q.weight",   (256, 256), 12),  # Q4_K
        ("output.weight",         (128, 256), 0),   # F32
    ]
    sp = WORK / "smoke_synth.gguf"
    gguf_write_synth(sp, specs)
    scan = gguf_scan(sp)

    expected = {}
    for name, dims, tid in specs:
        n = 1
        for d in dims:
            n *= d
        _, blk, tsz = ggml_type_info(tid)
        expected[name] = ((n + blk - 1) // blk) * tsz

    got = {t["name"]: t["nbytes"] for t in scan["tensors"]}
    res["checks"]["gguf_roundtrip_bytes_match"] = (got == expected)
    res["checks"]["gguf_expected"] = expected
    res["checks"]["gguf_got"] = got

    def bpw_of(name):
        t = [x for x in scan["tensors"] if x["name"] == name][0]
        return t["nbytes"] * 8 / t["n_elements"]

    res["checks"]["bpw_iq2_xs"] = round(bpw_of("blk.0.ffn_gate.weight"), 6)
    res["checks"]["bpw_iq4_xs"] = round(bpw_of("blk.0.ffn_up.weight"), 6)
    res["checks"]["bpw_iq2_xs_ok"] = abs(res["checks"]["bpw_iq2_xs"] - 2.3125) < 1e-6
    res["checks"]["bpw_iq4_xs_ok"] = abs(res["checks"]["bpw_iq4_xs"] - 4.25) < 1e-6

    # ---- 2. P1-G：未知 type 必须 fail closed ----
    bad = WORK / "smoke_unknown.gguf"
    gguf_write_synth(bad, [("blk.0.bogus.weight", (256, 256), 200)], strict_types=False)
    try:
        try:
            gguf_scan(bad)
            res["checks"]["unknown_type_fails_closed"] = False
            res["checks"]["unknown_type_error"] = "(没有抛异常，这是缺陷)"
        except UnknownGGMLType as e:
            res["checks"]["unknown_type_fails_closed"] = True
            res["checks"]["unknown_type_error"] = str(e)[:80]
        except Exception as e:
            res["checks"]["unknown_type_fails_closed"] = False
            res["checks"]["unknown_type_error"] = f"{type(e).__name__}: {e}"

        # 显式降级时：不产出字节，也不产出 bpw（0.0 bpw 同样是假数）
        try:
            sc_bad = gguf_scan(bad, strict_types=False)
            res["checks"]["unknown_type_degrades_to_none"] = (
                sc_bad["unknown_types"] == ["UNKNOWN"]
                and sc_bad["tensors"][0]["nbytes"] is None
                and sc_bad["bpw_effective"] is None)
            res["checks"]["unknown_type_degrades_detail"] = {
                "unknown_types": sc_bad["unknown_types"],
                "nbytes": sc_bad["tensors"][0]["nbytes"],
                "bpw_effective": sc_bad["bpw_effective"]}
        except Exception as e:
            res["checks"]["unknown_type_degrades_to_none"] = False
            res["checks"]["unknown_type_degrades_detail"] = f"{type(e).__name__}: {e}"
    finally:
        if bad.exists():
            bad.unlink()

    # ---- 3. P0-B：需求字节算术自检（防止优先级回归）----
    arith = {
        "8GiB+1GiB": _est_need_bytes(8 << 30, 1.0) == (9 << 30),
        "16GiB+2GiB": _est_need_bytes(16 << 30, 2.0) == (18 << 30),
        "8GiB+2GiB": _est_need_bytes(8 << 30, 2.0) == (10 << 30),
    }
    res["checks"]["need_bytes_arithmetic_ok"] = all(arith.values())
    res["checks"]["need_bytes_detail"] = arith
    res["checks"]["need_bytes_8gib_plus_2"] = _est_need_bytes(8 << 30, 2.0) / 2**30

    # ---- 4. P1-F：分组组数自检 ----
    grp_ok = True
    grp_detail = {}
    for n_paras, n_groups in [(5, 8), (7, 3), (8, 8), (9, 8), (40, 8), (41, 8), (100, 16)]:
        tmp = WORK / f"smoke_split_{n_paras}_{n_groups}"
        txt = "\n\n".join(f"para{i} " + ("w " * 40) for i in range(n_paras))
        tp = WORK / "smoke_split.txt"
        tp.write_text(txt)
        try:
            paths, meta = split_eval(tp, n_groups, tmp)
            exp = min(n_groups, n_paras)
            body = "".join(p.read_text() for p in paths)
            total = sum(m["paras"] for m in meta)
            ok = (len(paths) == exp and total == n_paras)
            grp_detail[f"{n_paras}p/{n_groups}g"] = f"{len(paths)}（期望 {exp}）"
            grp_ok = grp_ok and ok
        finally:
            if tp.exists():
                tp.unlink()
    res["checks"]["split_eval_exact_groups"] = grp_ok
    res["checks"]["split_eval_detail"] = grp_detail

    # ---- 5. P0-C：bf16 缺失时 _bench_summary 不能崩 ----
    try:
        s_ok = _bench_summary({"fp16": {"tflops": 50.0},
                               "bf16": {"error": "RuntimeError: not implemented for BFloat16"}})
        res["checks"]["bench_summary_bf16_error_ok"] = (
            s_ok["bf16_available"] is False
            and s_ok["ratio_bf16_over_fp16"] is None
            and "not implemented" in (s_ok["bf16_error"] or ""))
        s_no = _bench_summary({})
        res["checks"]["bench_summary_empty_ok"] = s_no["fp16_tflops"] is None
    except Exception as e:
        res["checks"]["bench_summary_bf16_error_ok"] = False
        res["checks"]["bench_summary_error"] = f"{type(e).__name__}: {e}"

    # ---- 6. GPU 微基准 ----
    tb = _torch_microbench(deadline)
    res["bench"]["torch"] = tb
    res["bench"]["torch_summary"] = _bench_summary(tb)
    res["bench"]["disk"] = _disk_bench(deadline)

    # ---- 7. 预算换算 ----
    res["budget_math"] = {
        "weights_iq2_xs_gib_lower_bound": round(CFG["PARAM_COUNT"] * 2.3125 / 8 / 2**30, 3),
        "weights_iq4_xs_gib_lower_bound": round(CFG["PARAM_COUNT"] * 4.25 / 8 / 2**30, 3),
        "weights_f16_gib": round(CFG["PARAM_COUNT"] * 2 / 2**30, 3),
        "note": "GiB 下界，不含 embed/head 的高精度保留；真实字节由 S5 从文件读",
        "nll_tokens_planned": CFG["NLL_GROUPS"] * CFG["NLL_CHUNKS_PER_GROUP"] * CFG["NLL_CTX"],
    }

    ok = all(v for k, v in res["checks"].items() if isinstance(v, bool))
    res["smoke_ok"] = ok

    print("=" * 78)
    print("smoke checks:")
    for k, v in res["checks"].items():
        print(f"  {k:34s} {v}")
    print("bench:", json.dumps(res["bench"], indent=1, ensure_ascii=False, default=str))
    print("budget:", json.dumps(res["budget_math"], indent=1, ensure_ascii=False))
    print(f"SMOKE_OK = {ok}")
    print("=" * 78)

    if not ok:
        raise RuntimeError("smoke 自检失败，先修解析器/算术再用大文件")
    return res


s1 = run_stage("S1_SMOKE", stage_s1_smoke)


---

## 从 S2 开始：源码是自动拿的，模型不是

S2 之后每个阶段都先做 `need(...)` 检查，缺路径直接 `GuardError`，**不会**去下载模型。

### S2_BUILD —— 自动取源码 + build 目录在 working

`RUN_S2 = True` 时，S2 做四件事，全部在一次点击内：

```
1. 下载 https://github.com/ggml-org/llama.cpp/archive/<pinned>.tar.gz   36.6 MiB
2. 校验 sha256 == f565ee97…4d7f（预登记值，不符就拒绝解压）
3. 安全解压到 /kaggle/working/trlab/src/llama.cpp-<commit>/（拒绝 ../ 与符号链接）
4. cmake -S <src> -B /kaggle/working/trlab/build-llama -DGGML_NATIVE=ON \
        -DGGML_CUDA={ON|OFF} -DCMAKE_CUDA_ARCHITECTURES=75 \
        -DLLAMA_USE_PREBUILT_UI=OFF -DCMAKE_BUILD_TYPE=Release
   cmake --build <build> --config Release -j 4 \
        --target llama-imatrix llama-quantize llama-perplexity llama-cli
```

`/kaggle/input` 是**只读**的，所以 build 目录必须落在 `/kaggle/working`。`llama_bin()` 的查找顺序是
`CFG["LLAMA_BIN"]` → `WORK/build-llama/bin` → `<src>/build/bin` → `PATH`，S3–S8 都走这个函数。

**为什么是 tarball 而不是 `git clone`**：旧版 `ALLOW_NETWORK=True` 走的是
`git clone --depth 1`，拿的是**当时的 HEAD** —— 与"固定 commit"直接冲突。现在只有一条
联网路径：pinned tarball + 预登记 sha256。目录里没有 `.git`，所以 commit 的校验方式记为
`commit_source="pinned_tarball"`：它证明"这份 tar.gz 就是当初 pin 的那份"，但不等于 git 校验。

**重复 Run All 不会重下**：解压后的目录 + `src/pinned_source.json` 里的 sha256 一致就直接复用；
tarball 还在且 digest 一致就只重新解压。断点文件在 `trlab/ckpt/state.json`。

`CFG["BUILD_CUDA"]=False` 时只构建 CPU 版 —— **imatrix 和 quantize 都是 CPU 路径，不需要 GPU**。
nvcc 不可用或 host g++ 版本不兼容时，S2 会给一句"症状 → 处置"，不会把几十行 cmake 输出丢给你。

### S3_IMATRIX

```
llama-imatrix -m <IMATRIX_BASE> -f <calib.txt> -o <out>.partial -c 512 --chunks 32 -ngl <n>
```

`-ngl` 由**显存拟合判定**自动决定：`base 字节 + 1.0 GiB < 可用显存` 就 `-ngl 99`，否则 `-ngl 0`（纯 CPU）。

**这里用的是 `IMATRIX_BASE`，不是 `BASE_GGUF`**（v0.2.1）。留空则两者相同。8B 的 f16 是 15.25 GiB，放不进约 15.0 GiB 可用显存 → 会退到 CPU（慢几十倍）；Q8_0（约 8.1 GiB）能进显存。所以 T4 上的正确做法是：`IMATRIX_BASE` 指向 Q8_0，`BASE_GGUF` 仍指向 f16。imatrix 是逐张量、逐输入通道的统计量，两份 base 结构一致就能用；S4 会核对 architecture 与每个张量的形状。

产物先写 `.partial`，用 GGUF 解析器验证 `n_tensors > 0` 之后才 `os.replace` 成正式文件名。

### S4_QUANT

```
llama-quantize --imatrix <imatrix.gguf> <BASE_GGUF> <out>.partial IQ2_XS <threads>
llama-quantize --imatrix <imatrix.gguf> <BASE_GGUF> <out>.partial IQ4_XS <threads>
```

两个都用 `--imatrix`。`IQ4_XS` 不要求它，但**必须给**，否则比较不公平。

**base 必须是 f16/bf16。** 上游在 `llama_model_quantize_impl` 里对已量化的张量有：

```cpp
if (ggml_is_quantized(tensor->type) && !params->allow_requantize)
    throw std::runtime_error(format("requantizing from type %s is disabled", ...));
```

`tools/quantize/quantize.cpp` 的帮助文本写得很直白：
`--allow-requantize ... WARNING: this can severely reduce quality compared to quantizing from 16bit or 32bit!`
所以 `BASE_KIND=q8_0` 时 S4 会**先**在 Python 侧拒绝并给出可执行的替代方案；只做 smoke 才显式 `CFG["ALLOW_REQUANTIZE"]=True`，此时命令追加 `--allow-requantize`，产物 provenance 标 `requantized_pilot_only`。

`llama-quantize` 是 CPU 路径，15.25 GiB 的 f16 走的是**内存**不是显存，T4 实例的 RAM 够。

**复用条件（v0.2.1）**：只有 `<file>.provenance.json` 里的 `quant_fp`（base 指纹 + imatrix sha256 + 源码 commit + 格式 + 二次量化标记 + 量化参数）与当前完全一致才复用。指纹不符时旧文件被改名成 `<file>.stale-<fp8>` 再重建 —— 不会出现"换了 imatrix，S4 说重跑了，文件其实还是旧的"。

`run_cmd` 在 rc≠0 时抛 `CommandFailed`，所以失败不会被"上一次留下的非空文件"掩盖。IQ2_XS 失败时会额外提示 imatrix 相关原因；被上游拒绝二次量化时会提示 `--allow-requantize`。


In [ ]:
# =====================================================================
# S2 · 源码获取 + 编译
#
# 联网只有一条路径：下载 S2_PINNED 里的 tarball（36.6 MiB），校验预登记 sha256，
# 解压到 WORK/src/，再从 WORK/build-llama 编译。**不 clone master。**
# =====================================================================

def _http_download(url, dst, deadline, expect_bytes=None, timeout=60, chunk=1 << 20):
    """把 url 下到 dst（先写 .partial），边下边算 sha256。

    只用标准库 urllib —— 不依赖 curl/wget，Kaggle 镜像里一定可用。
    任何网络异常都转成 GuardError，消息里直接写"检查 Settings → Internet = On"，
    不把 URLError 的原始栈丢给用户。
    """
    import socket
    import urllib.error
    import urllib.request

    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    part = dst.with_name(dst.name + ".partial")
    if part.exists():
        part.unlink()

    h = hashlib.sha256()
    got = 0
    t0 = time.time()
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "trlab-kaggle/0.3"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            total = int(r.headers.get("Content-Length") or 0)
            if expect_bytes and total and total != expect_bytes:
                raise GuardError(
                    f"服务器报的 Content-Length={total}，与预登记的 {expect_bytes} 不符。"
                    "上游文件可能被替换，拒绝继续。")
            with open(part, "wb") as f:
                while True:
                    if time.time() > deadline:
                        raise BudgetExceeded(f"下载超预算：已下 {got / 2**20:.1f} MiB")
                    b = r.read(chunk)
                    if not b:
                        break
                    f.write(b)
                    h.update(b)
                    got += len(b)
                    if got % (8 << 20) < chunk:
                        rate = got / 2**20 / max(1e-6, time.time() - t0)
                        log(f"    已下载 {got / 2**20:.1f} MiB"
                            + (f" / {total / 2**20:.1f}" if total else "")
                            + f"  {rate:.1f} MiB/s")
    except (GuardError, BudgetExceeded):
        if part.exists():
            part.unlink()
        raise
    except (urllib.error.HTTPError, urllib.error.URLError,
            socket.timeout, OSError) as e:
        if part.exists():
            part.unlink()
        raise GuardError(
            f"下载源码失败：{type(e).__name__}: {e}\n"
            "  → 检查 Notebook 右侧 Settings 里 Internet 是否为 On；\n"
            "  → 或改用离线方式：把源码挂到 /kaggle/input，设 CFG['LLAMA_SRC'] 到含 "
            "CMakeLists.txt 的那一层。")
    os.replace(part, dst)
    return {"bytes": got, "sha256": h.hexdigest(), "path": str(dst),
            "seconds": round(time.time() - t0, 1)}


def _safe_extract(tar_path, dest):
    """解压 tarball 到 dest。拒绝绝对路径、`..`、链接与非普通文件。

    llama.cpp 的归档里全是普通文件；出现别的就是有人动过包，直接拒绝而不是
    "尽量解压"。
    """
    dest = Path(dest).resolve()
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tf:
        for m in tf.getmembers():
            name = m.name
            if name.startswith("/") or ".." in Path(name).parts:
                raise GuardError(f"归档里有可疑路径 {name!r}，拒绝解压")
            if m.issym() or m.islnk():
                raise GuardError(f"归档里有链接 {name!r}，拒绝解压")
            if not (m.isfile() or m.isdir()):
                raise GuardError(f"归档里有非普通文件 {name!r}，拒绝解压")
            tgt = (dest / name).resolve()
            if tgt != dest and dest not in tgt.parents:
                raise GuardError(f"归档条目越界 {name!r}，拒绝解压")
        try:
            tf.extractall(dest, filter="data")      # Python ≥ 3.12
        except TypeError:
            tf.extractall(dest)
    return dest


def _fetch_pinned_source(deadline):
    """下载 → 校验 sha256 → 安全解压 → 校验顶层目录名。返回 (src_dir, meta)。

    缓存优先级（重复 Run All 不会重下）：
      1. 已解压 + stamp 里的 sha256 与当前 pin 一致 → 完全离线复用；
      2. 已下载的 tarball digest 一致 → 只解压，不重下；
      3. 否则下载。
    """
    pin = S2_PINNED
    src_dir = SRC_CACHE / pin["dir"]
    stamp = SRC_CACHE / "pinned_source.json"
    tar_path = DL_DIR / f"llama.cpp-{pin['commit'][:12]}.tar.gz"

    st = _read_json(stamp)
    if (isinstance(st, dict) and st.get("sha256") == pin["sha256"]
            and (src_dir / "CMakeLists.txt").exists()):
        log(f"源码缓存命中：{src_dir}（不联网）")
        return src_dir, {"origin": "cache", "sha256_ok": True,
                         "url": pin["url"], "dir": pin["dir"],
                         "archive_sha256": st.get("sha256"),
                         "archive_bytes": st.get("bytes"),
                         "archive_cached": True}

    meta = None
    if tar_path.exists() and tar_path.stat().st_size == pin["bytes"]:
        d = sha256_of(tar_path, deadline)
        if d == pin["sha256"]:
            log(f"复用已下载的源码包 {tar_path.name}（digest 一致，不重下）")
            meta = {"origin": "cached_archive", "archive_sha256": d,
                    "archive_bytes": pin["bytes"]}
        else:
            log(f"!! 已下载的源码包 digest 不符（{d[:12]}…），删掉重下")
            tar_path.unlink()

    if meta is None:
        if not CFG["S2_AUTO_FETCH_SOURCE"]:
            raise GuardError(
                "需要下载源码，但 CFG['S2_AUTO_FETCH_SOURCE'] 是 False。"
                "把顶部 RUN_S2 改成 True 即可。")
        log(f"下载固定 commit 源码：{pin['url']}")
        log(f"  预期 {pin['bytes'] / 2**20:.1f} MiB，sha256 {pin['sha256'][:16]}…")
        d = _http_download(pin["url"], tar_path, deadline,
                           expect_bytes=pin["bytes"])
        if d["sha256"] != pin["sha256"]:
            tar_path.unlink()
            raise GuardError(
                "源码包 sha256 不匹配，拒绝解压：\n"
                f"  预期 {pin['sha256']}\n"
                f"  实际 {d['sha256']}\n"
                "  → 上游文件可能被替换，或网络返回了错误页面（比如登录页）。"
                "不要继续。")
        log(f"  sha256 校验通过（{d['bytes'] / 2**20:.1f} MiB，{d['seconds']} s）")
        meta = {"origin": "downloaded", "archive_sha256": d["sha256"],
                "archive_bytes": d["bytes"]}

    SRC_CACHE.mkdir(parents=True, exist_ok=True)
    tmp = SRC_CACHE / f".extract-{os.getpid()}-{int(time.time())}"
    if tmp.exists():
        shutil.rmtree(tmp, ignore_errors=True)
    _safe_extract(tar_path, tmp)
    inner = tmp / pin["dir"]
    if not (inner / "CMakeLists.txt").exists():
        got = sorted(p.name for p in tmp.iterdir())[:5]
        shutil.rmtree(tmp, ignore_errors=True)
        raise GuardError(
            f"归档顶层目录不是预期的 {pin['dir']}（实际 {got}）。"
            "sha256 对得上但目录名不符，说明预登记信息本身有问题，不要继续。")
    if src_dir.exists():
        shutil.rmtree(src_dir, ignore_errors=True)
    os.replace(inner, src_dir)
    shutil.rmtree(tmp, ignore_errors=True)

    write_json(stamp, {"sha256": pin["sha256"], "bytes": pin["bytes"],
                       "commit": pin["commit"], "dir": pin["dir"],
                       "url": pin["url"],
                       "extracted_at": datetime.now(timezone.utc).isoformat(
                           timespec="seconds")})
    log(f"源码就位：{src_dir}")
    return src_dir, {"origin": meta["origin"], "sha256_ok": True,
                     "url": pin["url"], "dir": pin["dir"],
                     "archive_sha256": pin["sha256"],
                     "archive_bytes": pin["bytes"],
                     "archive_cached": meta["origin"] == "cached_archive"}


def _find_llama_src(deadline):
    """返回 {"src", "origin", "sha256_ok", ...}。

    优先级：手动（CFG["LLAMA_SRC"] / /kaggle/input 约定路径）> 自动获取。
    v0.3 删掉了旧分支"ALLOW_NETWORK=True 就 git clone --depth 1 master" ——
    那个分支拿的是 HEAD，与固定 LLAMA_COMMIT 直接冲突，等于让"固定版本"这句话
    失效，属于设计错误而不是缺少功能。
    """
    p = _manual_llama_src()
    if p:
        return {"src": p, "origin": "manual", "sha256_ok": False,
                "url": None, "dir": None,
                "archive_sha256": None, "archive_bytes": None}
    if CFG["S2_AUTO_FETCH_SOURCE"]:
        src_dir, meta = _fetch_pinned_source(deadline)
        out = {"src": src_dir}
        out.update(meta)
        return out
    raise GuardError(
        "找不到 llama.cpp 源码，且没有打开自动获取。\n"
        "  → 一键：把 notebook 顶部 RUN_S2 改成 True（需要 Settings 里 "
        "Internet = On，只下载 36.6 MiB 源码，不下载模型）。\n"
        "  → 离线：把源码挂到 /kaggle/input，并把 CFG['LLAMA_SRC'] 指到含 "
        "CMakeLists.txt 的那一层。")


def stage_s2_build(deadline):
    got = _find_llama_src(deadline)
    src = got["src"]
    git_commit = _git_commit(src)

    if git_commit:
        commit, commit_source, commit_verified = git_commit, "git", True
    elif got.get("sha256_ok"):
        # 目录里没有 .git，但归档 digest 与预登记值一致 —— 这比"声明值"强，
        # 比"git 校验"弱：它证明"这份 tar.gz 就是当初 pin 的那份"。
        commit = CFG["LLAMA_COMMIT"] or S2_PINNED["commit"]
        commit_source, commit_verified = "pinned_tarball", True
    elif CFG["LLAMA_COMMIT"]:
        commit, commit_source, commit_verified = CFG["LLAMA_COMMIT"], "declared", False
        log(f"!! 源码不是 git 仓库、也没有归档 digest，commit 记为声明值 {commit}（未校验）")
    else:
        commit, commit_source, commit_verified = None, None, False

    if CFG["LLAMA_COMMIT"] and commit and not commit.startswith(CFG["LLAMA_COMMIT"][:12]):
        raise GuardError(
            f"源码 commit {commit} 与预注册的 {CFG['LLAMA_COMMIT']} 不符。"
            "同一份报告不能混用两个 llama.cpp 版本。")

    # P0-A：build 必须落在可写目录。/kaggle/input 是只读的。
    BUILD.mkdir(parents=True, exist_ok=True)
    src_writable = os.access(src, os.W_OK)
    log(f"源码: {src}（来源={got.get('origin')}，可写={src_writable}）  build: {BUILD}")

    # 工具链预检：nvcc 缺失时在这里报错，别让 cmake 的 FATAL_ERROR 淹掉
    tc = _toolchain_report()
    log(f"工具链: {json.dumps(tc, ensure_ascii=False)}")
    pf = _cuda_preflight(tc, CFG["BUILD_CUDA"])
    if not pf["ok"]:
        raise GuardError(pf["note"])

    targets = ["llama-imatrix", "llama-quantize", "llama-perplexity", "llama-cli"]

    def _locate(n):
        for c in (BUILD / "bin" / n, BUILD / n):
            if c.exists():
                return c
        return None

    source = {"origin": got.get("origin"), "url": got.get("url"),
              "dir": got.get("dir"),
              "archive_sha256": got.get("archive_sha256"),
              "archive_bytes": got.get("archive_bytes"),
              "archive_cached": got.get("archive_cached", False)}

    have = {n: _locate(n) for n in targets}
    if all(v is not None for v in have.values()):
        log("已有完整构建，跳过编译")
        ver = subprocess.run([str(have["llama-cli"]), "--version"],
                             capture_output=True, text=True).stdout.strip()[:300]
        return {"src": str(src), "src_writable": src_writable, "build": str(BUILD),
                "reused": True, "commit": commit, "commit_source": commit_source,
                "commit_verified": commit_verified,
                "version": ver, "toolchain": tc, "source": source,
                "cuda": CFG["BUILD_CUDA"], "cuda_arch": CFG["CUDA_ARCH"],
                "cuda_no_vmm": bool(CFG.get("CUDA_NO_VMM") and CFG["BUILD_CUDA"]),
                "disk_free_gb": round(disk_gb(WORK)["free"], 2),
                "logs": {"cmake": str(LOGS / "s2_cmake.log"),
                         "build": str(LOGS / "s2_build.log")},
                "bins": {k: str(v) for k, v in have.items()}}

    cmake_args = ["cmake", "-S", str(src), "-B", str(BUILD),
                  "-DCMAKE_BUILD_TYPE=Release",
                  "-DGGML_NATIVE=ON",
                  f"-DGGML_CUDA={'ON' if CFG['BUILD_CUDA'] else 'OFF'}",
                  # 当前 master 已把 LLAMA_CURL 标为 deprecated 并忽略；保留只为兼容旧 pin
                  "-DLLAMA_CURL=OFF",
                  "-DLLAMA_BUILD_TESTS=OFF",
                  "-DLLAMA_BUILD_EXAMPLES=OFF",
                  # 否则 tools/ui 会在构建期 file(DOWNLOAD) 去 HF bucket 取 UI，
                  # Internet Off 时会白等几十秒到几分钟（不会失败，但纯浪费）
                  "-DLLAMA_USE_PREBUILT_UI=OFF"]
    if CFG["BUILD_CUDA"] and CFG["CUDA_ARCH"]:
        # 不指定时上游默认 50-v/61-v/70-v/75-v/80-v/86-r/89-r/90-v/120a-r 九套架构，
        # 4 核上编译量翻几倍。T4 只需要 sm_75。
        cmake_args.append(f"-DCMAKE_CUDA_ARCHITECTURES={CFG['CUDA_ARCH']}")
    if CFG["BUILD_CUDA"] and CFG.get("CUDA_NO_VMM"):
        # v0.3.1：Kaggle T4 真实失败 —— find_package(CUDAToolkit) 解析不出
        # CUDA::cuda_driver IMPORTED target，ggml-cuda 在
        # ggml/src/ggml-cuda/CMakeLists.txt:181 的 target_link_libraries 直接
        # CMake Error（Generate step failed）。上游 178-182 行有
        # if (GGML_CUDA_NO_VMM) 空分支 / else 链 cuda_driver，option 在
        # ggml/CMakeLists.txt:205 默认 OFF。打开它只跳过 libcuda 直链与 VMM
        # 显存池（回退 ggml_cuda_pool_leg，即 cudaMalloc 池），算力内核不变。
        cmake_args.append("-DGGML_CUDA_NO_VMM=ON")

    try:
        run_cmd(cmake_args, LOGS / "s2_cmake.log", deadline)
        run_cmd(["cmake", "--build", str(BUILD), "--config", "Release",
                 "-j", str(CFG["BUILD_JOBS"]), "--target", *targets],
                LOGS / "s2_build.log", deadline)
    except CommandFailed as e:
        raise RuntimeError(_explain_s2_failure(e)) from e

    bins = {n: _locate(n) for n in targets}
    missing = [k for k, v in bins.items() if v is None]
    if missing:
        raise RuntimeError(
            f"编译结束但缺少可执行文件 {missing}。"
            f"最可能的原因：有人关掉了 LLAMA_BUILD_SERVER（cli 在它的 if 里）。"
            f"看 {LOGS}/s2_build.log 最后 30 行。")

    ver = subprocess.run([str(bins["llama-cli"]), "--version"],
                         capture_output=True, text=True).stdout.strip()[:300]
    return {"src": str(src), "src_writable": src_writable, "build": str(BUILD),
            "reused": False, "commit": commit, "commit_source": commit_source,
            "commit_verified": commit_verified, "version": ver,
            "cuda": CFG["BUILD_CUDA"], "cuda_arch": CFG["CUDA_ARCH"],
            "cuda_no_vmm": bool(CFG.get("CUDA_NO_VMM") and CFG["BUILD_CUDA"]),
            "cmake_args": [str(a) for a in cmake_args], "toolchain": tc,
            "source": source,
            "disk_free_gb": round(disk_gb(WORK)["free"], 2),
            "logs": {"cmake": str(LOGS / "s2_cmake.log"),
                     "build": str(LOGS / "s2_build.log")},
            "bins": {k: str(v) for k, v in bins.items()}}


def _explain_s2_failure(exc):
    """把 cmake/nvcc 的失败压成"症状 → 处置"一句话，而不是把几十行日志丢给人。

    分类必须**同行匹配**：v0.3.0 用 `"nvcc" in tail and "not found" in tail`
    这种全文包含判断，结果 Kaggle 上"CUDA::cuda_driver target was not found"
    被误判成"找不到 nvcc"，还劝用户关 GPU —— 而 nvcc 明明检测通过了。
    """
    tail = (exc.tail or "").lower()
    lines = [l for l in tail.splitlines() if l.strip()]

    def _has(needle):
        return needle in tail

    def _same_line(a, b):
        return any(a in l and b in l for l in lines)

    if _has("unsupported gnu version") or _has("unsupported host compiler"):
        why = ("nvcc 不支持镜像里的 host g++ 版本。"
               "处置：CFG['BUILD_CUDA']=False（三个工具都有 CPU 路径，只是慢）。")
    elif _has("cuda::cuda_driver") or (_has("cuda_driver") and _has("target_link_libraries")):
        # 必须排在 nvcc 分支之前：这条错误里往往同时出现 nvcc 与 "not found"
        why = ("CMake 找不到 CUDA::cuda_driver IMPORTED target —— 镜像里 nvcc / "
               "CUDA runtime 都在，但 find_package(CUDAToolkit) 的 driver 组件没解析出来。"
               "v0.3.1 起默认传 -DGGML_CUDA_NO_VMM=ON 绕开（只跳过 libcuda 直链与 VMM "
               "显存池，算力内核不变）；如果你把 CFG['CUDA_NO_VMM'] 关成了 False，"
               "改回 True 再跑。")
    elif (_has("could not find cmake_cuda_compiler")
          or _has("no cmake_cuda_compiler")
          or _same_line("nvcc", "not found")
          or _same_line("nvcc", "could not find")
          or _same_line("nvcc", "no nvcc")):
        why = "找不到 nvcc。处置：CFG['BUILD_CUDA']=False，或换带 CUDA 的镜像。"
    elif _has("no space left"):
        why = ("磁盘满。处置：先删 trlab/dl 里的源码包和 trlab/build-llama 里没用的对象；"
               "别把 base 放进 /kaggle/working。")
    elif _has("killed") or _has("out of memory"):
        why = "编译时内存不足。处置：把 CFG['BUILD_JOBS'] 降到 2 再跑。"
    elif _has("cmake error"):
        why = ("cmake 配置阶段失败。处置：看日志最后一屏的 CMake Error 那几行；"
               "最常见的是源码目录指错层（要含 CMakeLists.txt 的那一层）。")
    else:
        why = "编译失败。处置：看日志最后 30 行。"
    return (f"S2 编译失败（rc={exc.rc}）。{why}\n"
            f"日志：{exc.log_path}\n"
            f"--- 日志尾部 ---\n{exc.tail}")


def _s2_report_lines(r):
    """S9 里复用的 S2 摘要（结果字典 → markdown 行）。"""
    L = []
    src = r.get("source") or {}
    L.append(f"- 源码：`{r.get('src')}`（来源 `{src.get('origin')}`）")
    if src.get("archive_sha256"):
        cached = "，复用已下载的包" if src.get("archive_cached") else ""
        L.append(f"- 归档 sha256：`{src['archive_sha256'][:16]}…` 与预登记值**一致**"
                 f"（{src.get('archive_bytes', 0) / 2**20:.1f} MiB{cached}）")
    L.append(f"- commit：`{r.get('commit')}`，来源 `{r.get('commit_source')}`，"
             f"校验 = **{'通过' if r.get('commit_verified') else '未通过/未校验'}**")
    L.append(f"- 编译复用：{r.get('reused')}；CUDA：{r.get('cuda')}"
             + (f"（arch {r.get('cuda_arch')}）" if r.get("cuda_arch") else "")
             + f"；NO_VMM：{r.get('cuda_no_vmm')}"
             + f"；磁盘剩余 {r.get('disk_free_gb')} GiB")
    tc = r.get("toolchain") or {}
    L.append(f"- 工具链：cmake {tc.get('cmake')}；nvcc {tc.get('nvcc')}；"
             f"g++ {tc.get('gxx')}")
    L.append("- 四个二进制：")
    for k, v in (r.get("bins") or {}).items():
        L.append(f"  - `{k}` → `{v}`")
    lg = r.get("logs") or {}
    L.append(f"- 日志：`{lg.get('cmake')}`、`{lg.get('build')}`")
    return L


s2 = run_stage("S2_BUILD", stage_s2_build)
if s2:
    print("-" * 78)
    print("S2 摘要")
    for _line in _s2_report_lines(s2):
        print(_line)
    print("-" * 78)
elif (STATE.get("S2_BUILD") or {}).get("status") == "error":
    print("-" * 78)
    print("S2 失败。上面那行 [S2_BUILD] FAIL 就是结论，细节在：")
    print(f"  {OUT / 'S2_BUILD.json'}")
    print(f"  {LOGS / 's2_cmake.log'}  /  {LOGS / 's2_build.log'}")
    print("S9 仍会照常生成报告，其余阶段因为开关是 False 不会启动。")
    print("-" * 78)


In [ ]:
def stage_s3_imatrix(deadline):
    # v0.2.1：imatrix 的 base 与量化的 base 可以不同（T4 上 imatrix 用 Q8_0，量化用 f16）
    base = _imatrix_base()
    base_kind = _imatrix_base_kind()
    calib = need("CALIB_TXT")
    if not base_kind:
        raise GuardError(
            "CFG['BASE_KIND'] 与 CFG['IMATRIX_BASE_KIND'] 都没设置"
            "（必须写 f16 / bf16 / q8_0 ...）")

    out_path = Path(CFG["IMATRIX_GGUF"]) if CFG["IMATRIX_GGUF"] else (MDLS / "imatrix.gguf")
    parent = out_path.parent
    parent.mkdir(parents=True, exist_ok=True)
    if not os.access(parent, os.W_OK):
        raise GuardError(
            f"{parent} 不可写。CFG['IMATRIX_GGUF'] 不能指向 /kaggle/input（只读）；"
            "留空则写到 working。")

    if calib.stat().st_size < 200_000:
        log(f"!! 校准语料只有 {calib.stat().st_size/1024:.0f} KB，imatrix 可能不稳")

    v = vram_gb()
    usable = (v["total"] - 1.0) if v else 0.0
    ngl = _fit_ngl(base.stat().st_size, usable)
    log(f"base={base.name} {base.stat().st_size/2**30:.2f} GiB, "
        f"可用显存 {usable:.1f} GiB → -ngl {ngl}")

    part = partial_path(out_path)
    if part.exists():
        part.unlink()

    cmd = [llama_bin("llama-imatrix"),
           "-m", base, "-f", calib, "-o", part,
           "-c", CFG["IMATRIX_CTX"], "--chunks", CFG["IMATRIX_CHUNKS"],
           "-ngl", ngl, "-t", os.cpu_count() or 4]
    run_cmd(cmd, LOGS / "s3_imatrix.log", deadline)

    # 验证后才原子改名（P1-E）
    if not part.exists() or part.stat().st_size < 100_000:
        raise RuntimeError(f"imatrix 产物缺失或过小: {part} "
                           f"({part.stat().st_size if part.exists() else 'missing'} B)")
    sc = gguf_scan(part)
    if sc["n_tensors"] == 0:
        raise RuntimeError(f"imatrix 产物是合法 GGUF 但没有任何张量: {part}")

    atomic_replace(part, out_path)

    rec = {
        "imatrix": str(out_path),
        "bytes": out_path.stat().st_size,
        "n_tensors": sc["n_tensors"],
        "sha256": sha256_of(out_path, deadline) if CFG["HASH_GGUF"] else None,
        "base_gguf": str(base),
        "base_kind": base_kind,
        "base_bytes": base.stat().st_size,
        "base_fp": _path_fp(base),
        "base_is_quantized": str(base_kind).lower() in QUANTIZED_KINDS,
        "imatrix_base_from_approx": str(base_kind).lower() in QUANTIZED_KINDS,
        "calib_txt": str(calib),
        "calib_bytes": calib.stat().st_size,
        "calib_source": CFG["CALIB_SOURCE"],
        "ctx": CFG["IMATRIX_CTX"],
        "chunks": CFG["IMATRIX_CHUNKS"],
        "ngl": ngl,
        "on_gpu": ngl > 0,
    }
    if rec["imatrix_base_from_approx"]:
        log(f"!! imatrix 由 {base_kind} 算的：激活是对 f16 的近似。"
            "这比『从 Q8_0 二次量化』好得多，但要在报告里声明。")
    print(json.dumps(rec, indent=1, ensure_ascii=False, default=str))
    return rec


s3 = run_stage("S3_IMATRIX", stage_s3_imatrix)


In [ ]:
FORMATS = ["IQ2_XS", "IQ4_XS"]


def _provenance_path(dst):
    """产物旁的 provenance 文件。文件名带 .gguf，所以不能用 with_suffix 替换。"""
    dst = Path(dst)
    return dst.with_name(dst.name + ".provenance.json")


def stage_s4_quant(deadline):
    base = need("BASE_GGUF")
    imat = resolve_imatrix()
    if imat is None:
        raise GuardError(
            "找不到 imatrix。IQ2_XS 没有 imatrix 会被 llama-quantize 直接 bail out，"
            "所以这里不做降级：先跑 S3，或把 CFG['IMATRIX_GGUF'] 指向已有文件。")

    # ---- P0-1：已量化的 base 必须显式 opt-in（否则上游直接抛） ----
    plan = _base_requant_plan(CFG["BASE_KIND"], CFG["ALLOW_REQUANTIZE"])
    if plan.get("status") == "blocked":
        raise GuardError(plan["reason"])
    if plan.get("flags"):
        log(f"!! 二次量化已放行（{CFG['BASE_KIND']} → {'/'.join(FORMATS)}）：{plan.get('caveat')}")

    # ---- imatrix 的 base 与量化的 base 是否同源 ----
    imat_base = _imatrix_base()
    if str(imat_base) == str(base):
        consistency = {"ok": True, "why": "同一个文件", "n_tensors": None,
                       "architecture": None}
    else:
        consistency = _base_consistency(gguf_scan(imat_base, deadline),
                                        gguf_scan(base, deadline))
        if consistency.get("ok") is False:
            raise GuardError(
                "IMATRIX_BASE 与 BASE_GGUF 不是同一个 checkpoint："
                f"{consistency['why']}。imatrix 是逐张量、逐输入通道的统计量，"
                "形状对不上就没有意义。")
        log(f"imatrix base {imat_base.name} 与量化 base {base.name} 结构一致"
            f"（{consistency.get('n_tensors')} 个张量，arch={consistency.get('architecture')}）")

    qbin = llama_bin("llama-quantize")
    threads = os.cpu_count() or 4
    base_fp = _path_fp(base)
    im_sha = sha256_of(imat, deadline) if CFG["HASH_GGUF"] else None
    llm_commit = (_git_commit(_llama_src_path()) or _git_commit(BUILD)
                  or (CFG["LLAMA_COMMIT"] or None))
    s3r = _get("S3_IMATRIX")

    # 决定字节的参数都进指纹（线程数不影响结果，不放）
    params = {"imatrix_ctx": CFG["IMATRIX_CTX"], "imatrix_chunks": CFG["IMATRIX_CHUNKS"]}

    out = {}
    for fmt in FORMATS:
        check_deadline(deadline, f"S4_QUANT {fmt}")
        dst = _model_path(fmt)
        prov_path = _provenance_path(dst)
        fp = _quant_fp(base_fp, im_sha, llm_commit, fmt, plan, params)
        prov = _read_json(prov_path)

        # P0-2：只有 provenance 完全匹配才复用。文件存在 ≠ 文件是对的。
        reuse = (dst.exists() and dst.stat().st_size > 0
                 and isinstance(prov, dict) and prov.get("quant_fp") == fp
                 and not CFG["FORCE_RERUN"])

        if reuse:
            log(f"{fmt} 复用：产物指纹 {fp} 与当前 base/imatrix/commit 一致")
        else:
            if dst.exists():
                why = ("CFG['FORCE_RERUN']=True" if CFG["FORCE_RERUN"] else
                       ("没有 provenance 记录，无法证明它来自当前输入"
                        if not isinstance(prov, dict) else
                        f"指纹变化 {(prov or {}).get('quant_fp')} → {fp}"))
                retired = _retire_product(dst, tag=(prov or {}).get("quant_fp") or fp)
                log(f"{fmt} 不复用（{why}）；旧产物 → {retired or '已删除（0 字节）'}")

            part = partial_path(dst)
            if part.exists():
                part.unlink()

            cmd = _quant_cmd(qbin, imat, base, part, fmt, threads, plan)
            try:
                run_cmd(cmd, LOGS / f"s4_quant_{fmt}.log", deadline)
            except CommandFailed as e:
                hint = ""
                low = (e.tail or "").lower()
                if "requantizing" in low:
                    hint = ("  ← 上游拒绝二次量化。BASE_KIND 是量化类型时必须显式设 "
                            "CFG['ALLOW_REQUANTIZE']=True，或改用 f16/bf16 的 BASE_GGUF。")
                elif "imatrix" in low or "garbage" in low or "importance matrix" in low:
                    hint = "  ← 这是 imatrix 缺失/无效触发的 bail out"
                raise RuntimeError(f"{fmt} 量化失败（rc={e.rc}）。{hint}") from e

            if not part.exists() or part.stat().st_size == 0:
                raise RuntimeError(f"{fmt} 量化后没有产物: {part}")
            atomic_replace(part, dst)

            write_json(prov_path, {
                "schema": 1,
                "fmt": fmt,
                "quant_fp": fp,
                "notebook": NOTEBOOK_VERSION,
                "base": str(base), "base_kind": CFG["BASE_KIND"], "base_fp": base_fp,
                "imatrix": str(imat), "imatrix_sha256": im_sha,
                "imatrix_base": str(imat_base), "imatrix_base_kind": _imatrix_base_kind(),
                "imatrix_base_consistent": consistency,
                "llama_commit": llm_commit,
                "requant": plan.get("provenance_tag"),
                "validity_scope": plan.get("validity_scope"),
                "params": params,
                "cmd": [str(c) for c in cmd],
                "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            })

        # 无论复用还是新产出，都重新验一遍
        sc = gguf_scan(dst, deadline)
        if sc["n_tensors"] == 0:
            raise RuntimeError(f"{fmt} 产物没有任何张量: {dst}")
        dom = max(sc["bytes_per_type"].items(), key=lambda x: x[1])[0] if sc["bytes_per_type"] else None
        if dom != fmt:
            log(f"!! {fmt} 的主导类型是 {dom}，不是 {fmt} —— 检查量化是否真的生效")

        out[fmt] = {
            "path": str(dst),
            "file_bytes": sc["file_bytes"],
            "file_gb": sc["file_gb"],
            "n_params": sc["n_params"],
            "bpw_effective": sc["bpw_effective"],
            "dominant_type": dom,
            "reused": bool(reuse),
            "quant_fp": fp,
            "provenance": str(prov_path),
            "sha256": sha256_of(dst, deadline) if CFG["HASH_GGUF"] else None,
        }
        log(f"{fmt}: {sc['file_gb']:.2f} GiB, {sc['bpw_effective']} bpw, "
            f"主导类型 {dom}, {'复用' if reuse else '新产出'}")

    # ---- 公平性：不再硬编码，也不再只看"同一个文件" ----
    if isinstance(s3r, dict):
        same_imatrix = (s3r.get("sha256") == im_sha) if im_sha else None
        imat_base_same = (str(s3r.get("base_gguf")) == str(base))
        origin = "S3" if s3r.get("imatrix") == str(imat) else "user_supplied"
        imatrix_base_kind = s3r.get("base_kind")
    else:
        same_imatrix, imat_base_same, origin, imatrix_base_kind = None, None, "user_supplied", None

    imatrix_base_kind = imatrix_base_kind or _imatrix_base_kind()
    out["_fairness"] = {
        "quant_base": str(base),
        "quant_base_kind": CFG["BASE_KIND"],
        "quant_base_fp": base_fp,
        "imatrix": str(imat),
        "imatrix_sha256": im_sha,
        "imatrix_base": str(imat_base),
        "imatrix_base_kind": imatrix_base_kind,
        "imatrix_base_same_as_quant_base": imat_base_same,
        "imatrix_base_consistent": consistency,
        "imatrix_base_from_approx": str(imatrix_base_kind).lower() in QUANTIZED_KINDS,
        "same_imatrix": same_imatrix,
        "imatrix_origin": origin,
        "base_scope": plan.get("validity_scope"),
        "requant": plan.get("provenance_tag"),
        "llama_commit": llm_commit,
        "verified": bool(same_imatrix) and consistency.get("ok") is not False,
        "note": ("verified 需要：imatrix 与 S3 记录的 sha256 一致，且 imatrix base 与量化 base "
                 "结构一致。None 表示无法验证（S3 未运行 / 未开 HASH_GGUF）。"
                 "base_scope=requantized_pilot_only 表示 base 本身是量化文件，"
                 "IQ2/IQ4 之间可比，但不能当作相对原始 checkpoint 的高保真结论。"),
    }
    print(json.dumps(out, indent=1, ensure_ascii=False, default=str))
    return out


s4 = run_stage("S4_QUANT", stage_s4_quant)


In [ ]:
def stage_s5_bytes(deadline):
    targets = {}
    if CFG["BASE_GGUF"] and Path(CFG["BASE_GGUF"]).exists():
        targets[f"base_{CFG['BASE_KIND'] or 'unknown'}"] = Path(CFG["BASE_GGUF"])
    for fmt in FORMATS:
        p = _model_path(fmt)
        if p.exists():
            targets[fmt] = p
    if not targets:
        raise GuardError("没有可分析的 GGUF。先跑 S4，或设置 CFG['BASE_GGUF']。")

    rows, detail, unavailable = [], {}, []
    for name, p in targets.items():
        check_deadline(deadline, f"S5 {name}")
        try:
            sc = gguf_scan(p, deadline, strict_types=True)
        except UnknownGGMLType as e:
            # P1-G：拒绝把未知类型按 8 bpw 写进有效结果
            log(f"!! {name}: {e}")
            unavailable.append({"name": name, "path": str(p), "reason": str(e)})
            detail[name] = {"byte_breakdown_available": False, "path": str(p),
                            "file_bytes": p.stat().st_size,
                            "file_gb": round(p.stat().st_size / 2**30, 3),
                            "error": str(e)}
            continue

        detail[name] = {
            "byte_breakdown_available": True,
            "file_bytes": sc["file_bytes"],
            "file_gb": sc["file_gb"],
            "n_params": sc["n_params"],
            "bpw_effective": sc["bpw_effective"],
            "bpw_filesize": sc["bpw_filesize"],
            "bytes_per_type": sc["bytes_per_type"],
            "bytes_per_class": sc["bytes_per_class"],
            "header_bytes": sc["header_bytes"],
            "n_tensors": sc["n_tensors"],
        }
        rows.append((name, sc["file_gb"], sc["bpw_effective"],
                     sc["bytes_per_class"].get("embed/head", 0), sc["n_params"]))

    print("=" * 78)
    print(f"{'name':22s} {'file GiB':>9s} {'bpw':>8s} {'embed+head GiB':>15s}")
    print("-" * 78)
    for name, gb, bpw, eh, _n in rows:
        bpw_s = f"{bpw:8.4f}" if bpw is not None else f"{'n/a':>8s}"
        print(f"{name:22s} {gb:9.3f} {bpw_s} {eh/2**30:15.3f}")
    print("-" * 78)
    for name, d in detail.items():
        if not d.get("byte_breakdown_available", False):
            print(f"{name}: 字节拆分不可用（未知 ggml type），不出 bpw")
            continue
        print(f"{name} 按类型:", {k: round(v / 2**30, 3) for k, v in d["bytes_per_type"].items()})
        if d["n_params"] == 0:
            print(f"  !! {name} 没有张量（只有词表/元数据），bpw 无法计算")
    print("=" * 78)

    ratios = {}
    d2, d4 = detail.get("IQ2_XS", {}), detail.get("IQ4_XS", {})
    if d2.get("byte_breakdown_available") and d4.get("byte_breakdown_available"):
        b2, b4 = d2["file_bytes"], d4["file_bytes"]
        t2 = b2 - d2["header_bytes"]
        t4 = b4 - d4["header_bytes"]
        ratios = {
            "iq4_over_iq2": round(b4 / b2, 4),
            "bytes_saved_pct": round((1 - b2 / b4) * 100, 2),
            "tensor_only_iq4_over_iq2": round(t4 / t2, 4),
            "note": "文件字节比，不是端到端加速比",
        }
        print("IQ4_XS / IQ2_XS 字节比:", ratios)

    return {"per_model": detail, "ratios": ratios,
            "byte_breakdown_unavailable": unavailable}


s5 = run_stage("S5_BYTES", stage_s5_bytes)


## S6 · 配对 NLL

同一个 base、同一份 imatrix、同一批 token 窗口，逐组配对。组间 token 数不严格相等，所以同时给字符加权均值。

In [ ]:
def stage_s6_nll(deadline):
    eval_txt = need("EVAL_TXT")
    if not CFG["EVAL_SOURCE"]:
        log("!! CFG['EVAL_SOURCE'] 未设置，provenance 里会缺这一项")

    # ---- 公平性：S4 的核验结果决定这次比较能不能叫"公平" ----
    s4r = _get("S4_QUANT") or {}
    fair = s4r.get("_fairness") or {}
    if fair.get("same_imatrix") is False:
        raise RuntimeError(
            "S4 记录显示 imatrix 的 sha256 与 S3 不一致，配对比较无效。重跑 S4。")
    cons = fair.get("imatrix_base_consistent") or {}
    if cons.get("ok") is False:
        raise RuntimeError(
            f"imatrix 的 base 与量化 base 不是同一个 checkpoint（{cons.get('why')}），"
            "配对比较无效。")
    fairness_verified = bool(fair.get("verified"))
    base_scope = fair.get("base_scope")
    scope_note = None
    if base_scope == "requantized_pilot_only":
        scope_note = ("base 本身是量化文件（已显式 ALLOW_REQUANTIZE）：IQ2 与 IQ4 都从它再量化，"
                      "两者之间的配对差仍然同源可比，但**不能**作为"
                      "『相对原始 checkpoint BF16』的高保真低比特结论。")
        log("!! " + scope_note)
    elif fair.get("imatrix_base_from_approx"):
        scope_note = ("量化 base 是高精度，但 imatrix 由量化过的 base 算出："
                      "激活是对高精度的近似，影响远小于二次量化，仍需在结论里声明。")

    # ---- 校准/评测隔离断言（内容级）----
    isolation = {"ok": None}
    if CFG["CALIB_TXT"] and Path(CFG["CALIB_TXT"]).exists():
        c = Path(CFG["CALIB_TXT"]).read_text(errors="replace")
        e = eval_txt.read_text(errors="replace")
        cset = {ln.strip()[:200] for ln in c.split("\n") if len(ln.strip()) > 40}
        eset = {ln.strip()[:200] for ln in e.split("\n") if len(ln.strip()) > 40}
        ov = cset & eset
        isolation.update(
            calib=str(CFG["CALIB_TXT"]), calib_source=CFG["CALIB_SOURCE"],
            eval=str(eval_txt), eval_source=CFG["EVAL_SOURCE"],
            calib_lines=len(cset), eval_lines=len(eset),
            overlap_lines=len(ov), ok=(len(ov) == 0))
        if ov:
            raise RuntimeError(
                f"校准集与评测集有 {len(ov)} 行重叠（前 200 字符比对）。"
                "必须换 split 后再跑，否则 NLL 会被 imatrix 污染。")
        log(f"隔离检查通过：calib {len(cset)} 行 / eval {len(eset)} 行 / 重叠 0")
    else:
        log("!! 没有校准语料路径，无法做隔离断言 —— 结论里必须标注这一点")

    models = {f: _model_path(f) for f in FORMATS if _model_path(f).exists()}
    if len(models) < 2:
        raise GuardError(f"需要两个格式的文件，现有 {list(models)}。先跑 S4。")

    groups, meta = split_eval(eval_txt, CFG["NLL_GROUPS"], WORK / "eval_groups")
    log(f"评测集切成 {len(groups)} 组：" +
        ", ".join(f"g{m['group']}({m['paras']}段/{m['chars']}字符)" for m in meta))

    per_model = {}
    for fmt, mp in models.items():
        per_model[fmt] = []
        for i, g in enumerate(groups):
            check_deadline(deadline, f"S6 {fmt} g{i}")
            r = run_ppl(mp, g, deadline, f"{fmt}_g{i}")
            r["group"] = i
            r["chars"] = meta[i]["chars"]
            per_model[fmt].append(r)
            log(f"  {fmt} g{i}: PPL={r['ppl']:.4f} NLL={r['nll']:.4f}")

    d = [per_model["IQ2_XS"][i]["nll"] - per_model["IQ4_XS"][i]["nll"]
         for i in range(len(groups))]
    stats = paired_bootstrap(d)

    # 组间字符数不等时，均值口径会有偏；给一个字符加权版本作对照
    w = [m["chars"] for m in meta]
    tot_w = sum(w) or 1
    stats["char_weighted_mean"] = sum(di * wi for di, wi in zip(d, w)) / tot_w

    agg = {}
    for fmt, vals in per_model.items():
        mu = sum(x["nll"] for x in vals) / len(vals)
        agg[fmt] = {
            "mean_nll": mu,
            "mean_ppl": sum(x["ppl"] for x in vals) / len(vals),
            "nll_sd": (sum((x["nll"] - mu) ** 2 for x in vals) / max(1, len(vals) - 1)) ** 0.5,
        }

    out = {
        "isolation": isolation,
        "fairness_verified": fairness_verified,
        "fairness_record": fair,
        "base_scope": base_scope,
        "scope_note": scope_note,
        "eval_source": CFG["EVAL_SOURCE"],
        "groups": len(groups),
        "group_meta": meta,
        "per_group": per_model,
        "aggregate": agg,
        "paired_delta_nll_iq2_minus_iq4": stats,
        "deltas": [round(x, 6) for x in d],
        "statistics_note": ("NLL_GROUPS 只用于 pilot 估计。bootstrap 的 n 是**组数**不是 token 数，"
                            "组间 token 数也不严格相等；正式 CI 应按独立文档重新设计。"),
        "provenance": {
            "base_gguf": CFG["BASE_GGUF"], "base_kind": CFG["BASE_KIND"],
            "imatrix": str(resolve_imatrix()),
            "imatrix_base": (fair or {}).get("imatrix_base"),
            "base_scope": base_scope,
            "self_quantized": True,
            "fair_comparison": fairness_verified,
            "note": "同 base、同 imatrix、同 token 窗口；fair_comparison=False 表示来源未核验",
        },
    }

    print("=" * 78)
    for fmt, a in agg.items():
        print(f"{fmt:10s} mean NLL={a['mean_nll']:.4f}  mean PPL={a['mean_ppl']:.4f}  sd={a['nll_sd']:.4f}")
    print(f"配对差 (IQ2 - IQ4): mean={stats['mean']:+.4f}  "
          f"char_weighted={stats['char_weighted_mean']:+.4f}  "
          f"CI95={stats['ci95']}  n={stats['n']}")
    print(f"公平性已核验: {fairness_verified}")
    print("=" * 78)

    ci = stats.get("ci95")
    if ci and ci[0] > 0:
        verdict = "IQ2_XS 显著差于 IQ4_XS（CI 全在 0 以上）"
    elif ci and ci[1] < 0:
        verdict = "IQ2_XS 显著好于 IQ4_XS（异常，先查口径）"
    else:
        verdict = "CI 跨 0 —— 未测出差异，不等于等价"
    out["verdict"] = verdict
    print("判读:", verdict)
    return out


s6 = run_stage("S6_NLL", stage_s6_nll)


In [ ]:
QUALITY_PROMPTS_DEFAULT = [
    "The capital of France is",
    "def fibonacci(n):",
    "In 1969, the first humans landed on",
    "The chemical formula for table salt is",
    "Q: What is 17 * 23?\nA:",
    "Once upon a time, in a small village by the sea,",
    "The three primary colors of light are",
    "Translate 'good morning' into Chinese:",
]


def probe_help(binary):
    try:
        p = subprocess.run([str(binary), "--help"], capture_output=True, text=True, timeout=90)
        return (p.stdout or "") + (p.stderr or "")
    except Exception:
        return ""


def _cli_flags():
    """llama-cli 的 flag 名跨版本变过，这里探测而不是硬编码。"""
    h = probe_help(llama_bin("llama-cli"))
    return {
        "no_conv": ("--no-conversation" if "--no-conversation" in h
                    else ("-no-cnv" if "-no-cnv" in h else None)),
        "no_display_prompt": "--no-display-prompt" if "--no-display-prompt" in h else None,
        "temp": "--temp" if "--temp" in h else "-temp",
        "seed": "--seed" if "--seed" in h else "-s",
        "simple_io": "--simple-io" if "--simple-io" in h else None,
    }


def repeat_ratio(text, n=4):
    toks = text.split()
    if len(toks) < n + 1:
        return 0.0
    grams = [tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)]
    return 1.0 - len(set(grams)) / len(grams)


def stage_s7_quality(deadline):
    flags = _cli_flags()
    log(f"llama-cli flags: {flags}")
    cli = llama_bin("llama-cli")

    models = {f: _model_path(f) for f in FORMATS if _model_path(f).exists()}
    if len(models) < 2:
        raise GuardError("需要两个格式的文件，先跑 S4")

    prompts = QUALITY_PROMPTS_DEFAULT[:CFG["QUALITY_PROMPTS"]]
    v = vram_gb()
    usable = (v["total"] - 1.0) if v else 0.0

    per_model = {}
    for fmt, mp in models.items():
        ngl = _fit_ngl(mp.stat().st_size, usable)
        per_model[fmt] = []
        for i, pr in enumerate(prompts):
            check_deadline(deadline, f"S7 {fmt} p{i}")
            cmd = [cli, "-m", mp, "-p", pr, "-n", CFG["QUALITY_N_PREDICT"],
                   flags["temp"], "0", flags["seed"], CFG["SEED"],
                   "-ngl", ngl, "-t", os.cpu_count() or 4]
            for key in ("no_conv", "no_display_prompt", "simple_io"):
                if flags.get(key):
                    cmd.append(flags[key])
            logf = LOGS / f"s7_{fmt}_p{i}.log"
            run_cmd(cmd, logf, deadline)          # rc != 0 直接抛，不再当成功

            txt = logf.read_text(errors="replace")
            body = txt.split(pr, 1)[1] if pr in txt else txt
            body = body.replace("llama_perf", "\nllama_perf").split("\nllama_perf")[0].strip()
            per_model[fmt].append({"prompt": pr, "text": body[:400],
                                   "repeat4": round(repeat_ratio(body), 4),
                                   "chars": len(body)})

    pairs = []
    for i, pr in enumerate(prompts):
        a = per_model["IQ2_XS"][i]["text"]
        b = per_model["IQ4_XS"][i]["text"]
        k = 0
        while k < min(len(a), len(b)) and a[k] == b[k]:
            k += 1
        pairs.append({"prompt": pr, "identical": a == b, "first_divergence_char": k,
                      "len_iq2": len(a), "len_iq4": len(b)})

    n_id = sum(1 for p in pairs if p["identical"])
    out = {
        "n_prompts": len(prompts),
        "identical_outputs": n_id,
        "identical_rate": round(n_id / len(prompts), 4) if prompts else None,
        "pairs": pairs,
        "per_model": per_model,
        "degenerate": {fmt: round(sum(x["repeat4"] for x in v2) / len(v2), 4)
                       for fmt, v2 in per_model.items()},
        "limits": [
            "贪心解码 + 32 token，样本量很小，只能发现整体崩塌，不能排序细微质量",
            "不是标准任务评测（GSM8K / HumanEval 之类），绝对分数不构成能力证据",
            "repeat4 只是退化信号，不是质量分数",
        ],
    }
    print("=" * 78)
    print(f"完全一致的输出: {n_id}/{len(prompts)}  ({out['identical_rate']})")
    print(f"平均 repeat4: {out['degenerate']}")
    for p in pairs:
        print(f"  div@{p['first_divergence_char']:4d}  {p['prompt'][:44]}")
    print("=" * 78)
    return out


s7 = run_stage("S7_QUALITY", stage_s7_quality)


## S8 · 高精度锚点

先算显存，再判精度分档。**单张 T4 16 GB 装不下 8B f16**；Q8_0 只算近似参照，默认拒绝。

In [ ]:
def stage_s8_bf16_anchor(deadline):
    """高精度锚点。默认在 T4 上会被显存判定或精度分档拒绝。"""
    anchor = need("BASE_GGUF")
    v = vram_gb()
    total = v["total"] if v else 0.0

    d = _anchor_decision(
        anchor_bytes=anchor.stat().st_size,
        vram_total_gb=total,
        base_kind=CFG["BASE_KIND"],
        allow_approx=CFG["ANCHOR_ALLOW_APPROX"],
        extra_gb=CFG["ANCHOR_EXTRA_GB"],
    )
    d["anchor_path"] = str(anchor)
    d["anchor_dtype_note"] = (
        "T4 是 CC 7.5，没有原生 bf16。锚点请用 f16；用 bf16 文件会走转换路径，"
        "与论文口径不同，必须声明。"
        if str(CFG["BASE_KIND"]).lower() == "bf16" else
        f"锚点 dtype = {CFG['BASE_KIND']}；与论文常用的 BF16 锚点不完全一致，跨平台比较时必须声明。")

    if d["status"] != "ready":
        print(json.dumps(d, indent=1, ensure_ascii=False, default=str))
        return d

    eval_txt = need("EVAL_TXT")
    groups, meta = split_eval(eval_txt, CFG["NLL_GROUPS"], WORK / "eval_groups")
    vals = []
    for i, g in enumerate(groups):
        check_deadline(deadline, f"S8 g{i}")
        r = run_ppl(anchor, g, deadline, f"anchor_g{i}")
        r["group"] = i
        r["chars"] = meta[i]["chars"]
        vals.append(r)
        log(f"  anchor g{i}: PPL={r['ppl']:.4f}")

    d.update(
        status="ok",
        per_group=vals,
        mean_nll=sum(x["nll"] for x in vals) / len(vals),
        mean_ppl=sum(x["ppl"] for x in vals) / len(vals),
        note=("锚点 NLL。validity_scope=high_precision 时才能算 "
              "rho = dNLL(Q2,Q4)/dNLL(Q4,anchor) 并当有效性证据；"
              "approx_only 时只能当量级参考。"),
    )
    print(json.dumps({k: v2 for k, v2 in d.items() if k != "per_group"},
                     indent=1, ensure_ascii=False, default=str))
    return d


s8 = run_stage("S8_BF16_ANCHOR", stage_s8_bf16_anchor)


In [ ]:
def stage_sx_explore(deadline):
    """第三方现成量化：只看字节和 NLL，**永远不进入公平比较表**。"""
    p = need("EXPLORE_GGUF")
    try:
        sc = gguf_scan(p, deadline)
        breakdown_ok = True
        info = {"file_gb": sc["file_gb"], "bpw_effective": sc["bpw_effective"],
                "bytes_per_type": sc["bytes_per_type"]}
    except UnknownGGMLType as e:
        breakdown_ok = False
        info = {"file_gb": round(p.stat().st_size / 2**30, 3),
                "bpw_effective": None, "error": str(e)}

    rec = {
        "path": str(p),
        "byte_breakdown_available": breakdown_ok,
        **info,
        "provenance": {
            "self_quantized": False,
            "fair_comparison": False,
            "why_not": ("第三方 GGUF 的校准集、imatrix、量化脚本、llama.cpp 版本全部未知。"
                        "它的 NLL 只能说明这个文件本身如何，不能与自建 IQ2_XS/IQ4_XS 做公平比较。"),
            "allowed_use": "探索性测试：确认格式可加载、量级是否合理、有没有明显崩塌",
        },
    }

    if CFG["EVAL_TXT"] and Path(CFG["EVAL_TXT"]).exists():
        groups, meta = split_eval(Path(CFG["EVAL_TXT"]), CFG["NLL_GROUPS"], WORK / "eval_groups")
        vals = []
        for i, g in enumerate(groups):
            check_deadline(deadline, f"SX g{i}")
            r = run_ppl(p, g, deadline, f"explore_g{i}")
            r["group"] = i
            vals.append(r)
        rec["mean_nll"] = sum(x["nll"] for x in vals) / len(vals)
        rec["mean_ppl"] = sum(x["ppl"] for x in vals) / len(vals)
        rec["per_group"] = vals
    else:
        rec["mean_nll"] = None
        rec["note"] = "没有评测语料，只做了字节分析"

    print(json.dumps({k: v2 for k, v2 in rec.items() if k != "per_group"},
                     indent=1, ensure_ascii=False, default=str))
    return rec


sx = run_stage("SX_EXPLORE", stage_sx_explore)


## S9 · 汇总报告

这个 cell **每次都会重跑**，不会因为之前成功过就 RESUME。

In [ ]:
def stage_s9_report(deadline):
    assert THIRD_PARTY_IN_COMPARISON is False, "第三方量化禁止进入公平比较表"

    env = _get("S0_ENV")
    smoke = _get("S1_SMOKE")
    s3r, s4r = _get("S3_IMATRIX"), _get("S4_QUANT")
    s5r, s6r = _get("S5_BYTES"), _get("S6_NLL")
    s7r, s8r, sxr = _get("S7_QUALITY"), _get("S8_BF16_ANCHOR"), _get("SX_EXPLORE")

    L = []
    L.append("# transformer-residual-lab · Kaggle 运行报告\n")
    L.append(f"- 生成时间（UTC）：{datetime.now(timezone.utc).isoformat(timespec='seconds')}")
    L.append(f"- Notebook 版本：v{NOTEBOOK_VERSION}")
    L.append(f"- 模型：`{CFG['MODEL_ID']}`（{CFG['PARAM_COUNT']/1e9:.2f} B 参数）\n")

    # ---- 1 环境 ----
    L.append("## 1. 环境\n")
    if env:
        L.append(f"- 位置：{env['where']}，Python {env['python']}")
        L.append(f"- CPU {env['cpu_count']} 核，RAM {env['ram']}")
        for g in env.get("gpus", []):
            L.append(f"- GPU{g['idx']}：{g['name']}，CC{g['cc']}，{g['total_gb']} GiB，"
                     f"native_bf16={g['native_bf16']}")
        L.append(f"- 可用显存估计：{env['vram_usable_gb_est']} GiB")
        for k, v2 in env.get("disks_gb", {}).items():
            L.append(f"- 磁盘 `{k}`：free {v2.get('free')} GiB")
        for w in env.get("warnings", []):
            L.append(f"- **注意**：{w}")
    else:
        L.append("_S0_ENV 未运行_")
    L.append("")

    # ---- 2 工具链（S2） ----
    L.append("## 2. 工具链（S2_BUILD）\n")
    _s2rec = STATE.get("S2_BUILD") or {}
    if _s2rec.get("result"):
        L.extend(_s2_report_lines(_s2rec["result"]))
    elif _s2rec.get("status") == "error":
        L.append(f"- **S2 失败**：`{str(_s2rec.get('error'))[:400]}`")
        L.append(f"- 结果文件：`{OUT}/S2_BUILD.json`")
        L.append(f"- 日志：`{LOGS}/s2_cmake.log`、`{LOGS}/s2_build.log`")
        L.append("- 其余阶段没有启动（`STAGES` 里是 False），所以报告里下面几节是空的。")
    else:
        L.append("_S2_BUILD 未运行（`RUN_S2 = False`）。这一节为空不代表失败。_")
    L.append("")

    # ---- 3 smoke ----
    L.append("## 3. Smoke test\n")
    if smoke:
        L.append(f"- SMOKE_OK = **{smoke.get('smoke_ok')}**")
        c = smoke.get("checks", {})
        L.append(f"- IQ2_XS bpw 自检：{c.get('bpw_iq2_xs')}（应为 2.3125）")
        L.append(f"- IQ4_XS bpw 自检：{c.get('bpw_iq4_xs')}（应为 4.25）")
        L.append(f"- 未知 type fail-closed：{c.get('unknown_type_fails_closed')}")
        L.append(f"- 需求字节算术（P0-B 回归）：{c.get('need_bytes_arithmetic_ok')}，"
                 f"8 GiB+2 GiB = {c.get('need_bytes_8gib_plus_2')} GiB")
        L.append(f"- 分组组数（P1-F 回归）：{c.get('split_eval_exact_groups')}")

        # P0-C：绝不假设 bf16 一定可用
        tb = (smoke.get("bench") or {}).get("torch") or {}
        bs = _bench_summary(tb)
        if bs.get("torch_error"):
            L.append(f"- GPU 微基准不可用：{bs['torch_error']}")
        elif bs.get("cuda") is False:
            L.append("- GPU 微基准：无 GPU，跳过")
        else:
            L.append(f"- fp16：{bs['fp16_tflops']} TFLOP/s")
            if bs["bf16_available"]:
                L.append(f"- bf16：{bs['bf16_tflops']} TFLOP/s"
                         f"（相对 fp16 比值 {bs['ratio_bf16_over_fp16']}）")
            else:
                L.append(f"- bf16：**不可用** —— {bs['bf16_error'] or '未测出'}")
                L.append("  → 这不是缺陷。T4（CC 7.5）没有原生 bf16，"
                         "所以锚点只能用 f16，且 bf16 相关结论一律不可引用。")
            if bs.get("copy_gb_per_s"):
                L.append(f"- 拷贝带宽（派生值，非 DRAM 计数器）：{bs['copy_gb_per_s']} GB/s")
        dsk = (smoke.get("bench") or {}).get("disk") or {}
        if dsk:
            L.append(f"- 磁盘写：{dsk.get('mb_per_s')} MB/s")
    else:
        L.append("_S1_SMOKE 未运行_")
    L.append("")

    # ---- 3 字节 ----
    L.append("## 4. 实际字节\n")
    scope = ((s4r or {}).get("_fairness") or {}).get("base_scope")
    if scope == "requantized_pilot_only":
        L.append("> **base 本身是二次量化**（`BASE_KIND` 是量化类型，已显式打开 "
                 "`ALLOW_REQUANTIZE`）。IQ2/IQ4 之间的配对差仍然同源可比，"
                 "但**不能**作为『相对原始 checkpoint BF16』的高保真低比特结论。\n")
    if s5r:
        L.append("| 文件 | 文件 GiB | 有效 bpw | embed+head GiB | 拆分可用 |")
        L.append("|---|---|---|---|---|")
        for name, d in s5r["per_model"].items():
            ok = d.get("byte_breakdown_available", False)
            if ok:
                eh = d["bytes_per_class"].get("embed/head", 0) / 2**30
                bpw = f"{d['bpw_effective']:.4f}" if d["bpw_effective"] is not None else "n/a"
                L.append(f"| {name} | {d['file_gb']:.3f} | {bpw} | {eh:.3f} | 是 |")
            else:
                L.append(f"| {name} | {d['file_gb']:.3f} | — | — | **否**（未知 ggml type） |")
        if s5r.get("ratios"):
            L.append("")
            L.append(f"字节比（IQ4/IQ2）：{s5r['ratios'].get('iq4_over_iq2')}，"
                     f"节省 {s5r['ratios'].get('bytes_saved_pct')}%")
        if s5r.get("byte_breakdown_unavailable"):
            L.append("")
            L.append("**字节拆分不可用的文件**（不产出 bpw，避免假数）：")
            for u in s5r["byte_breakdown_unavailable"]:
                L.append(f"- `{u['name']}`：{u['reason'][:90]}")
    else:
        L.append("_S5_BYTES 未运行_")
    L.append("")

    # ---- 4 NLL ----
    L.append("## 5. 配对 NLL\n")
    if s6r:
        iso = s6r.get("isolation", {})
        L.append(f"- 评测语料来源：{s6r.get('eval_source') or '未声明'}")
        L.append(f"- 校准/评测隔离：{iso.get('ok')}（重叠 {iso.get('overlap_lines')} 行）")
        L.append(f"- **公平性已核验**：{s6r.get('fairness_verified')}")
        L.append(f"- 分组数：{s6r['groups']}，每组 {CFG['NLL_CHUNKS_PER_GROUP']} chunk × "
                 f"{CFG['NLL_CTX']} ctx")
        L.append("")
        L.append("| 格式 | 平均 NLL | 平均 PPL | NLL 标准差 |")
        L.append("|---|---|---|---|")
        for fmt, a in s6r["aggregate"].items():
            L.append(f"| {fmt} | {a['mean_nll']:.4f} | {a['mean_ppl']:.4f} | {a['nll_sd']:.4f} |")
        st = s6r["paired_delta_nll_iq2_minus_iq4"]
        L.append("")
        L.append(f"配对差（IQ2 − IQ4）：均值 **{st['mean']:+.4f} nats**，"
                 f"字符加权 {st.get('char_weighted_mean', float('nan')):+.4f}，"
                 f"95% CI {st['ci95']}，n={st['n']}")
        L.append(f"\n判读：{s6r.get('verdict')}")
        if s6r.get("scope_note"):
            L.append(f"\n**结论效力**：{s6r['scope_note']}")
        L.append(f"\n统计口径限制：{s6r.get('statistics_note')}")
    else:
        L.append("_S6_NLL 未运行_")
    L.append("")

    # ---- 5 质量 ----
    L.append("## 6. 有限配对质量检查\n")
    if s7r:
        L.append(f"- 贪心续写完全一致：{s7r['identical_outputs']}/{s7r['n_prompts']}"
                 f"（{s7r['identical_rate']}）")
        L.append(f"- 平均 repeat4（越低越好）：{s7r['degenerate']}")
        for lim in s7r["limits"]:
            L.append(f"- 限制：{lim}")
    else:
        L.append("_S7_QUALITY 未运行_")
    L.append("")

    # ---- 6 锚点 ----
    L.append("## 7. 高精度锚点\n")
    if s8r:
        L.append(f"- 状态：**{s8r.get('status')}**")
        L.append(f"- 锚点精度：{s8r.get('base_kind')}（tier={s8r.get('tier')}）")
        L.append(f"- 需要 {s8r.get('required_gb')} GiB / 可用 {s8r.get('vram_usable_gb')} GiB")
        if s8r.get("status") != "ok":
            L.append(f"- 跳过原因：{s8r.get('reason')}")
        else:
            L.append(f"- 锚点 NLL：{s8r.get('mean_nll'):.4f}（{s8r.get('validity_scope')}）")
            if s6r and s8r.get("validity_scope") == "high_precision":
                rho = _rho_from_nll(
                    anchor_nll=s8r["mean_nll"],
                    q4_nll=s6r["aggregate"]["IQ4_XS"]["mean_nll"],
                    q2_nll=s6r["aggregate"]["IQ2_XS"]["mean_nll"])
                if rho["rho"] is not None:
                    L.append(f"- rho = dNLL(Q2,Q4)/dNLL(Q4,anchor) = {rho['rho']:.4f}")
                    L.append(f"  （dNLL(Q4,anchor)={rho['d4']:+.4f}，"
                             f"dNLL(Q2,anchor)={rho['d2']:+.4f}；均为「后减前」）")
                else:
                    L.append(f"- **不计算 rho**：{rho['reason']}")
            elif s6r:
                L.append("- **不计算 rho**：锚点是近似参照，不构成有效性证据。")
        L.append(f"- dtype 偏差声明：{s8r.get('anchor_dtype_note')}")
    else:
        L.append("_S8_BF16_ANCHOR 未运行_")
    L.append("")

    # ---- 7 第三方 ----
    L.append("## 8. 第三方量化（仅探索）\n")
    if sxr:
        L.append(f"- 文件：`{sxr['path']}`，{sxr['file_gb']} GiB，"
                 f"bpw={sxr.get('bpw_effective')}")
        L.append(f"- **不参与公平比较**：{sxr['provenance']['why_not']}")
    else:
        L.append("_SX_EXPLORE 未运行_")
    L.append("")

    # ---- 8 结论边界 ----
    L.append("## 9. 结论边界\n")
    L.append("1. 所有字节数字来自**文件读取**；未知 ggml type 的文件不出 bpw。")
    L.append("2. 配对 CI 基于**分组** bootstrap；**CI 跨 0 表示未测出差异，不是等价**。")
    L.append("3. `NLL_GROUPS` 只用于 pilot。组间 token 数不严格相等，正式 CI 需按独立文档重设计。")
    L.append("4. 单张 T4、8B、单一评测语料，**不能外推到 27B 或 M4 Pro**。")
    L.append("5. T4 无原生 bf16；锚点用 f16 时与论文口径有偏差，已在第 7 节声明。")
    L.append("6. Q8_0 之类属二次量化，**不能**作为『相对原始 checkpoint BF16』的有效性证据。"
             "S4 默认拒绝从量化 base 量化；显式 `ALLOW_REQUANTIZE=True` 时产物标 "
             "`requantized_pilot_only`。")
    L.append("7. 第三方量化不进比较表；进了就是缺陷（本 cell 开头有断言）。")
    L.append("8. `guard()` 只在阶段入口检查显存，**不能阻止外部进程在运行中 OOM**。")
    L.append("9. S4 只在产物 provenance 指纹（base + imatrix + commit + 参数）完全一致时复用旧 "
             "GGUF；指纹不符的旧文件被改名成 `.stale-*`，不会被静默当成新结果。")
    L.append("")

    md = "\n".join(L)
    (OUT / "report.md").write_text(md, encoding="utf-8")

    # 自一致性：写 results.json 时本阶段还没被 run_stage 登记进 STATE，
    # 直接取会是 null，与日志里的 "S9_REPORT OK" 矛盾。显式补上并标明是自报。
    stage_snapshot = {}
    for k in STAGES:
        rec = STATE.get(k) or {}
        stage_snapshot[k] = {"status": rec.get("status"),
                             "elapsed_sec": rec.get("elapsed_sec"),
                             "fingerprint": rec.get("fingerprint"),
                             "error": rec.get("error")}
    stage_snapshot["S9_REPORT"] = {
        "status": "ok",
        "elapsed_sec": None,
        "fingerprint": _fingerprint("S9_REPORT"),
        "error": None,
        "self_reported": True,
        "note": "本阶段在写盘时尚未登记进 STATE；status 由这里显式置为 ok",
    }

    results = {
        "notebook_version": NOTEBOOK_VERSION,
        "model": CFG["MODEL_ID"],
        "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "stages": stage_snapshot,
        "config_snapshot": {k: v2 for k, v2 in CFG.items() if k != "STAGE_BUDGET_SEC"},
        "s0_env": env, "s1_smoke": smoke,
        "s3_imatrix": s3r, "s4_quant": s4r, "s5_bytes": s5r,
        "s6_nll": s6r, "s7_quality": s7r, "s8_anchor": s8r, "sx_explore": sxr,
        "disclaimer": ("自建量化与第三方量化的结果不可混用；"
                       "所有 NLL 均为该文件在该语料上的观测，不构成能力结论。"),
    }
    write_json(OUT / "results.json", results)

    print(md)
    print("=" * 78)
    print("产物:")
    for p in sorted(OUT.iterdir()):
        print(f"  {p.stat().st_size:>9d} B  {p.name}")
    return {"report": str(OUT / "report.md"), "results": str(OUT / "results.json"),
            "stage_status": {k: v2["status"] for k, v2 in stage_snapshot.items()}}


s9 = run_stage("S9_REPORT", stage_s9_report)


In [ ]:
# 最终产物清单 —— 从这里下载，或点 Kaggle 右上角 "Save Version"
print("=" * 78)
for p in sorted(OUT.iterdir()):
    print(f"{p.stat().st_size:>10d} B  {p.name}")
print("-" * 78)
for p in sorted(MDLS.glob("*.gguf")):
    print(f"{p.stat().st_size/2**30:>10.3f} GiB  {p.name}")
leftover = list(MDLS.glob("*.partial")) + list(OUT.glob("*.partial"))
if leftover:
    print("!! 残留 .partial（说明上次有阶段中途失败）:")
    for p in leftover:
        print(f"   {p}")
stale = list(MDLS.glob("*.stale-*"))
if stale:
    print("旧产物（指纹不符被移走，可删）:")
    for p in stale:
        print(f"   {p.stat().st_size/2**30:>8.3f} GiB  {p.name}")
provs = sorted(MDLS.glob("*.provenance.json"))
if provs:
    print("产物 provenance:")
    for p in provs:
        rec = json.loads(p.read_text(encoding="utf-8"))
        print(f"   {p.name}  fp={rec.get('quant_fp')}  "
              f"base_kind={rec.get('base_kind')}  requant={rec.get('requant')}")
print("-" * 78)
print(f"断点文件: {STATE_PATH}")
print("阶段状态:")
for k in STAGES:
    r = STATE.get(k) or {}
    print(f"  {k:16s} switch={str(STAGES[k]):5s} status={r.get('status', '-'):5s} "
          f"fp={r.get('fingerprint', '-')}"
          f"{('  err: ' + str(r.get('error'))[:60]) if r.get('error') else ''}")
print("=" * 78)
print("提醒：/kaggle/working 只有约 20 GB。")
print("      两个 GGUF 约 2.2 + 4.0 GiB，imatrix 只有几十 MB；不要在 working 里再放 base。")


---

## 后续模块规划（本版未实现）

| 模块 | 内容 | 阻塞点 |
|---|---|---|
| **M1 位宽阶梯** | 同 base 再加 `Q4_K_M` / `Q6_K` / `Q8_0`，形成 5 点阶梯，用 `rho` 曲线定位 2.3 bpw 落在哪一段 | 磁盘与时间翻倍 |
| **M2 真 KL** | `llama-perplexity --kl-divergence-base <logits.npy>`，先跑锚点存 logits | 锚点在 T4 跑不了；8B × 8k token 的 logits 约 5 GB |
| **M3 任务级** | GSM8K 子集 / HumanEval+ / BFCL 格式子集 | 需挂载数据集；8B 在 2.3 bpw 上可能接近地板 |
| **M4 端到端 tok/s** | `llama-bench` 的 `pp512` / `tg128`，报兑现率 `R = (t4/t2)/(B4/B2)` | 需 CUDA 构建 + 足够长的测量窗口 |
| **M5 跨平台锚点** | 同 checkpoint 在 A100 40/80 上跑 BF16 锚点，与 T4 的 f16 锚点对齐 | 需第二张卡；**吞吐不可跨平台迁移，只有格式与质量可以** |

### 关于 M4 的一个提醒

`R = (t_4 / t_2) / (B_4 / B_2)`，理想情形 `R = 1`。**不要写反成 `(B_4/B_2)/(t_2/t_4)`** —— 那样理想情形会得到 `1.84² = 3.39`，任何结果都会"通过"。

而且 `R` 只是**带宽受限区间的上界兑现率**。压到极低 bytes/token 之后会转成 latency-bound，实际加速低于字节比。

---

## 已知限制

1. **T4 无原生 bf16（CC 7.5）**。S1 会实测并如实记录 bf16 是否可用；不可用时 S9 明确写"不可用"而不是崩。
2. **单张 T4 装不下 8B f16**（15.25 GiB 权重 vs 约 15.0 GiB 可用，且未计 KV）。S8 返回 `skipped` 并给出 `required_gb` / `vram_usable_gb`。
3. **Q8_0 锚点是二次量化**。`ANCHOR_ALLOW_APPROX=False` 时 S8 直接拒绝；打开后结果标 `validity_scope="approx_only"`，S9 不算 `rho`。
4. **隔离只到内容级**：前 200 字符的行级比对。同文档不同段落检测不出来 —— 必须靠 `CALIB_SOURCE` / `EVAL_SOURCE` 声明不同 split。
5. **`/kaggle/working` 约 20 GB**。base 只能只读挂载。
6. **Kaggle GPU 会话 9 小时**。`MAX_WALL_SEC` 默认 8 小时。
7. **S7 样本量极小**（8 prompt × 32 token），只能发现整体崩塌。
8. **`guard()` 不是 OOM 保护**：只在阶段入口检查。llama.cpp 是外部进程，运行中自己分配显存，Python 侧无法阻止。
9. **输入指纹不是密码学保证**：用 size + mtime + 首尾 1 MiB，能抓"换了 base / 换了 imatrix / 改了 chunk"，不能对抗刻意伪造。


---

## 在 Kaggle 上手动执行

1. **新建 Notebook** → `File` → `Import Notebook` → 上传 `trlab_kaggle.ipynb`
2. 右侧 `Settings`：`Accelerator` = **GPU T4 ×2**（只用第 0 张）；`Internet` = **Off**
3. `Run All`。默认只跑 `S0_ENV` / `S1_SMOKE` / `S9_REPORT`，约 2–4 分钟，**不联网、不下载**
4. 想连 llama.cpp 一起编好：把 `RUN_S2` 改成 `True`、`Internet` 设 **On**，再 `Run all`
   （S2 自己下载固定 commit 的 36.6 MiB 源码并校验 sha256，只此一处联网）
5. 跑 `S0_ENV`，从输出里抄出 `/kaggle/input/...` 的真实路径
6. 回配置 cell，填 `CFG`，把要跑的阶段改成 `True`
7. **从上到下逐段跑**
8. 最后跑 `S9_REPORT`，产物在 `/kaggle/working/trlab/out/`

`Add Input` 只用于 S3 起真正需要的东西：base GGUF、校准语料、评测语料。llama.cpp 源码**不需要**挂载。

### 只想先确认环境

保持只有 `S0_ENV` / `S1_SMOKE` / `S9_REPORT` 为 `True`，`Run All`。约 2–4 分钟，不碰大文件。

### 断点恢复怎么工作

```
trlab/ckpt/state.json      # 每阶段含 status / fingerprint / elapsed / error
trlab/out/<STAGE>.json     # 每阶段结果，独立可读
trlab/logs/*.log           # 外部命令完整输出
trlab/models/*.gguf        # 产物（.partial 是未完成的）
```

- 指纹一致 → `RESUME`
- 指纹变化 → `INVALIDATE` 并重跑该阶段
- `S9_REPORT` 永远重跑
- 想忽略一切重来：`CFG["FORCE_RERUN"] = True`（最后手段，会重算所有前置）


---

## 一句话总结

首版只做**三件不花钱的事**：把环境问清楚、把 GGUF 字节解析器和算术验证对、把预算算出来。**任何需要下载、需要大显存、需要长时间运行的东西，都必须你手动打开开关。**

如果 `S0_ENV` 说 `/kaggle/input` 下没有 GGUF，那 S3–S7 一定跑不了 —— 这是设计如此，不是 bug。

`S1_SMOKE` 里有五条回归断言（字节往返、未知 type fail-closed、需求字节算术、分组组数、bf16 缺失不崩），它们是防止 v0.1 那类问题再进来的闸门。
